# MPG-FER v2.2 — source-locked segmented Kaggle T4 candidate

Generated from Issue #95 reviewed sources. The scientific package is source-locked; this notebook adds only runtime gates, resume verification, and artifact finalization.


In [ ]:
# STAGING TOOL EDITS ONLY THESE EXECUTION VALUES.
RESUME_MODE = "fresh"       # "fresh", "auto", or "required"
RESUME_PATH = None           # optional explicit /kaggle/input/.../resume_latest.pt
SEGMENT_NUMBER = 1
OUTPUT_DIR = "/kaggle/working/mpg_fer_v2_2_run"
ACCOUNT = "irthn1311"
KERNEL_REF = "irthn1311/mpg-fer-v2-2-t4-review-candidate"
GIT_COMMIT_SHA = "SET_BY_STAGING"
EXPECTED_SOURCE_SHA = "cbdeee5d5336338115895d2484ab35c3b233c25718d6c03768b7e0f5a2e93cca"
EXPECTED_PARAMETERS = 2_304_528


In [ ]:
# Generated from reviewed Issue #95 sources. Do not hand-edit embedded code.
import sys
from pathlib import Path

EMBEDDED_SOURCES = {'config.py': '"""Configuration contract for MPG-FER v2.2 (Issue #95)."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\n\n@dataclass\nclass MPGConfig:\n    seed: int = 42\n    device: str = "cuda"\n\n    img_size: int = 48\n    num_pixels: int = 2304\n    num_neighbors: int = 8\n    raw_pixel_dim: int = 32\n    d_pixel: int = 96\n    num_pixel_gnn_layers: int = 4\n    num_pixel_heads: int = 4\n    pixel_dropout: float = 0.10\n    pixel_drop_path_max: float = 0.03\n    pixel_edge_dim: int = 5\n    d_pixel_readout: int = 128\n\n    num_motifs: int = 48\n    motif_window_sizes: tuple[int, int, int] = (8, 12, 16)\n    motif_window_size: int = 12\n    motif_stride: int = 6\n    num_occurrences: int = 49\n    d_what: int = 96\n    d_type: int = 32\n    d_where: int = 5\n    d_motif_occurrence_raw: int = 133\n    d_motif: int = 192\n    tau_start: float = 0.70\n    tau_final: float = 0.30\n    tau_anneal_end_epoch: int = 35\n\n    num_motif_layers: int = 5\n    num_motif_heads: int = 6\n    motif_geom_dim: int = 6\n    motif_dropout: float = 0.10\n    motif_drop_path_max: float = 0.05\n    d_motif_readout: int = 384\n    # Dynamic sparse motif routing schedule: K edges per query node for layers 1..5\n    motif_topk_schedule: tuple[int, ...] = (8, 16, 16, 16, 24)\n\n    d_classifier_in: int = 512\n    d_classifier_hidden: int = 256\n    num_classes: int = 7\n    classifier_dropout: float = 0.25\n    supcon_dim: int = 128\n    supcon_temperature: float = 0.10\n\n    aux_pixel_weight: float = 0.05\n    aux_motif_weight: float = 0.20\n    lambda_div: float = 0.01\n    lambda_mi: float = 0.025\n    mi_beta: float = 1.0\n    consistency_probability: float = 0.50\n    lambda_consistency: float = 0.15\n    lambda_supcon: float = 0.05\n\n    ema_decay: float = 0.999\n    batch_size: int = 16\n    gradient_accumulation_steps: int = 2\n    learning_rate: float = 3e-4\n    weight_decay: float = 5e-4\n    max_epochs: int = 120\n    min_epochs: int = 50\n    warmup_epochs: int = 5\n    lr_decay_end_epoch: int = 85\n    min_learning_rate: float = 1e-6\n    early_stop_monitor_start_epoch: int = 85\n    early_stop_patience: int = 15\n    grad_clip: float = 1.0\n    label_smoothing: float = 0.05\n    use_amp: bool = True\n    num_workers: int = 2\n\n    resume_schema_version: int = 3\n    resume_snapshot_interval: int = 10\n    resume_snapshots_to_keep: int = 2\n    segment_soft_limit_hours: float = 10.5\n    segment_safety_margin_minutes: float = 15.0\n    segment_number: int = 1\n    run_id: str | None = None\n    output_dir: str | None = None\n    resume_path: str | None = None\n\n    micro_overfit_samples: int = 16\n    micro_overfit_target: float = 0.875\n    micro_overfit_max_steps: int = 80\n    micro_overfit_learning_rate: float = 1e-3\n\n    runtime_safe_resume_fields: tuple[str, ...] = field(\n        default=(\n            "num_workers", "output_dir", "resume_path", "segment_number",\n            "segment_soft_limit_hours", "segment_safety_margin_minutes", "run_id",\n        ),\n        repr=False,\n    )\n\n    def __post_init__(self) -> None:\n        if len(self.motif_topk_schedule) != self.num_motif_layers:\n            raise ValueError(\n                "motif_topk_schedule length must equal num_motif_layers"\n            )\n        maximum = self.num_occurrences - 1\n        if any(\n            isinstance(value, bool)\n            or not isinstance(value, int)\n            or not 1 <= value <= maximum\n            for value in self.motif_topk_schedule\n        ):\n            raise ValueError(\n                f"motif_topk_schedule values must be integers in [1, {maximum}]"\n            )\n', 'features.py': '"""Continuous H1-inspired 32D relational pixel feature extraction."""\n\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass PixelFeatureExtractor(nn.Module):\n    """Extracts exactly 32 raw continuous relational features per pixel on a 48x48 image.\n\n    Feature composition per pixel:\n      - 1  normalized intensity I in [0, 1]\n      - 2  normalized coordinates x, y in [-1, 1]\n      - 24 sigma-normalized center-relative differences from 5x5 neighborhood\n      - 1  log_sigma: log(sqrt(var_local + eps^2) + eps)\n      - 4  gx, gy, gradient_magnitude, laplacian\n      ------------------------------------------------------------------------\n      Total: exactly 32 dimensions.\n\n    Uses reflection padding for the 5x5 spatial support to preserve all 2304 pixels.\n    """\n\n    def __init__(self, img_size: int = 48, eps: float = 1e-5) -> None:\n        super().__init__()\n        self.img_size = img_size\n        self.num_pixels = img_size * img_size\n        self.eps = eps\n\n        # Precompute normalized spatial coordinate grids in [-1, 1]\n        coords = torch.linspace(-1.0, 1.0, img_size)\n        grid_y, grid_x = torch.meshgrid(coords, coords, indexing="ij")\n        # [1, 1, 48, 48]\n        self.register_buffer("grid_x", grid_x.unsqueeze(0).unsqueeze(0).clone())\n        self.register_buffer("grid_y", grid_y.unsqueeze(0).unsqueeze(0).clone())\n\n        # Sobel gradient filters\n        sobel_x = torch.tensor(\n            [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]],\n            dtype=torch.float32,\n        ).reshape(1, 1, 3, 3) / 8.0\n        sobel_y = torch.tensor(\n            [[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]],\n            dtype=torch.float32,\n        ).reshape(1, 1, 3, 3) / 8.0\n        laplacian = torch.tensor(\n            [[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]],\n            dtype=torch.float32,\n        ).reshape(1, 1, 3, 3) / 4.0\n\n        self.register_buffer("sobel_x", sobel_x)\n        self.register_buffer("sobel_y", sobel_y)\n        self.register_buffer("laplacian", laplacian)\n\n        # 5x5 local mean filter\n        ones_5x5 = torch.ones((1, 1, 5, 5), dtype=torch.float32) / 25.0\n        self.register_buffer("mean_5x5", ones_5x5)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        """Args:\n\n            x: [B, 1, 48, 48] images normalized to [0, 1].\n\n        Returns:\n            features: [B, 2304, 32] continuous relational pixel features.\n        """\n        B, C, H, W = x.shape\n        assert C == 1 and H == self.img_size and W == self.img_size\n\n        # 1. Normalized intensity I: [B, 1, 48, 48]\n        feat_I = x\n\n        # 2. Normalized coordinates (x, y): [B, 2, 48, 48]\n        feat_coords = torch.cat(\n            [self.grid_x.expand(B, -1, -1, -1), self.grid_y.expand(B, -1, -1, -1)],\n            dim=1,\n        )\n\n        # 3. 5x5 neighborhood unfolding with reflection padding\n        # Pad by 2 on all sides to keep exact (48, 48) grid\n        x_pad = F.pad(x, (2, 2, 2, 2), mode="reflect")\n        # unfold: [B, 25, 48, 48]\n        patches = F.unfold(x_pad, kernel_size=5, padding=0, stride=1).reshape(\n            B, 25, H, W\n        )\n\n        # Center pixel is at index 12 (row 2, col 2 in 5x5)\n        center_pixel = patches[:, 12:13, :, :]  # [B, 1, H, W]\n\n        # 24 non-center neighbor pixels\n        indices_24 = [i for i in range(25) if i != 12]\n        neighbors_24 = patches[:, indices_24, :, :]  # [B, 24, H, W]\n\n        # Local variance & sigma over 5x5 window\n        mean_local = F.conv2d(x_pad, self.mean_5x5)\n        mean_sq_local = F.conv2d(x_pad**2, self.mean_5x5)\n        var_local = F.relu(mean_sq_local - mean_local**2)\n        sigma_local = torch.sqrt(var_local + (self.eps**2))  # [B, 1, H, W]\n\n        # 24 sigma-normalized center-relative differences:\n        # d_k = (I_neighbor - I_center) / sqrt(var_local + eps^2)\n        diffs_24 = (neighbors_24 - center_pixel) / sigma_local  # [B, 24, H, W]\n\n        # 4. log_sigma: log(sqrt(var_local + eps^2) + eps)\n        log_sigma = torch.log(sigma_local + self.eps)  # [B, 1, H, W]\n\n        # 5. Gradients & Laplacian with reflection padding\n        x_pad3 = F.pad(x, (1, 1, 1, 1), mode="reflect")\n        gx = F.conv2d(x_pad3, self.sobel_x)\n        gy = F.conv2d(x_pad3, self.sobel_y)\n        grad_mag = torch.sqrt(gx**2 + gy**2 + (self.eps**2))\n        lap = F.conv2d(x_pad3, self.laplacian)\n        # [B, 4, H, W]\n        derivatives = torch.cat([gx, gy, grad_mag, lap], dim=1)\n\n        # Concatenate all 32 channels:\n        # 1 (I) + 2 (coords) + 24 (diffs) + 1 (log_sigma) + 4 (derivatives) = 32\n        all_feats = torch.cat(\n            [feat_I, feat_coords, diffs_24, log_sigma, derivatives], dim=1\n        )  # [B, 32, 48, 48]\n\n        # Reshape to [B, 2304, 32]\n        # Channels-last: [B, 48, 48, 32] -> [B, 2304, 32]\n        all_feats = all_feats.permute(0, 2, 3, 1).reshape(B, self.num_pixels, 32)\n        return all_feats\n', 'graph.py': '"""Fixed 8-neighbor pixel graph topology and relative edge feature construction."""\n\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\n\n\nclass PixelGraphTopology(nn.Module):\n    """Builds and caches the static 8-neighbor directed graph over a 48x48 pixel grid.\n\n    Neighborhood layout for receiver i at (r_i, c_i):\n      0: NW (r-1, c-1)\n      1: N  (r-1, c)\n      2: NE (r-1, c+1)\n      3: W  (r,   c-1)\n      4: E  (r,   c+1)\n      5: SW (r+1, c-1)\n      6: S  (r+1, c)\n      7: SE (r+1, c+1)\n\n    Edges are directed receiver-sender relations: j -> i (sender j contributing to receiver i).\n    Edge features (5D):\n      [dx, dy, distance, delta_I, abs_delta_I]\n      where dx = x_j - x_i, dy = y_j - y_i, delta_I = I_j - I_i.\n    """\n\n    def __init__(self, img_size: int = 48) -> None:\n        super().__init__()\n        self.img_size = img_size\n        self.num_pixels = img_size * img_size\n        self.num_neighbors = 8\n\n        # 8 neighbor relative row, col offsets: (dr, dc)\n        # where r_j = r_i + dr, c_j = c_i + dc\n        self.neighbor_offsets = [\n            (-1, -1),  # 0: NW\n            (-1, 0),  # 1: N\n            (-1, 1),  # 2: NE\n            (0, -1),  # 3: W\n            (0, 1),  # 4: E\n            (1, -1),  # 5: SW\n            (1, 0),  # 6: S\n            (1, 1),  # 7: SE\n        ]\n\n        neighbor_idx = torch.zeros(\n            (self.num_pixels, self.num_neighbors), dtype=torch.long\n        )\n        neighbor_mask = torch.zeros(\n            (self.num_pixels, self.num_neighbors), dtype=torch.bool\n        )\n        geom_edges = torch.zeros(\n            (self.num_pixels, self.num_neighbors, 3), dtype=torch.float32\n        )\n\n        coords = torch.linspace(-1.0, 1.0, img_size)\n\n        for r_i in range(img_size):\n            for c_i in range(img_size):\n                i = r_i * img_size + c_i\n                x_i = coords[c_i].item()\n                y_i = coords[r_i].item()\n\n                for k, (dr, dc) in enumerate(self.neighbor_offsets):\n                    r_j = r_i + dr\n                    c_j = c_i + dc\n\n                    if 0 <= r_j < img_size and 0 <= c_j < img_size:\n                        j = r_j * img_size + c_j\n                        neighbor_idx[i, k] = j\n                        neighbor_mask[i, k] = True\n\n                        x_j = coords[c_j].item()\n                        y_j = coords[r_j].item()\n\n                        dx = x_j - x_i\n                        dy = y_j - y_i\n                        dist = (dx**2 + dy**2) ** 0.5\n\n                        geom_edges[i, k, 0] = dx\n                        geom_edges[i, k, 1] = dy\n                        geom_edges[i, k, 2] = dist\n                    else:\n                        neighbor_idx[i, k] = (\n                            i  # self-loop pad to avoid out-of-bounds gather\n                        )\n                        neighbor_mask[i, k] = False\n                        geom_edges[i, k] = 0.0\n\n        self.register_buffer("neighbor_idx", neighbor_idx)  # [2304, 8]\n        self.register_buffer("neighbor_mask", neighbor_mask)  # [2304, 8]\n        self.register_buffer("geom_edges", geom_edges)  # [2304, 8, 3]\n\n        # Pre-expanded indices for batched gathering [2304, 8, 1]\n        I_gather_idx = neighbor_idx.unsqueeze(-1)\n        self.register_buffer("I_gather_idx", I_gather_idx)\n\n    def compute_edge_features(self, pixel_intensity: torch.Tensor) -> torch.Tensor:\n        """Compute the 5D edge features for directed edges j -> i.\n\n        Args:\n            pixel_intensity: [B, 2304, 1] normalized pixel intensities.\n\n        Returns:\n            edge_features: [B, 2304, 8, 5]\n              [dx, dy, distance, delta_I, abs_delta_I]\n        """\n        B, N, _ = pixel_intensity.shape\n        assert N == self.num_pixels\n\n        # Expand static geometry: [1, 2304, 8, 3] -> [B, 2304, 8, 3]\n        geom = self.geom_edges.unsqueeze(0).expand(B, -1, -1, -1)\n\n        # Efficient flatten-gather across batch\n        # pixel_intensity: [B, 2304, 1]\n        # I_j for receiver i: gather sender neighbor index\n        I_j = pixel_intensity[:, self.neighbor_idx, :]  # [B, 2304, 8, 1]\n        I_i = pixel_intensity.unsqueeze(2)  # [B, 2304, 1, 1]\n\n        delta_I = I_j - I_i  # [B, 2304, 8, 1]\n        abs_delta_I = torch.abs(delta_I)\n\n        # Edge features: [dx, dy, dist, delta_I, abs_delta_I] -> [B, 2304, 8, 5]\n        edge_feats = torch.cat([geom, delta_I, abs_delta_I], dim=-1)\n\n        # Zero out invalid padded boundary edges\n        mask = self.neighbor_mask.unsqueeze(0).unsqueeze(-1)  # [1, 2304, 8, 1]\n        edge_feats = edge_feats * mask.float()\n        return edge_feats\n', 'losses.py': '"""Differentiable objectives introduced by MPG-FER v2.2."""\n\nfrom __future__ import annotations\n\nimport math\nimport torch\nimport torch.nn.functional as F\n\n\ndef motif_mutual_information_loss(\n    assignments: torch.Tensor, beta: float = 1.0, eps: float = 1e-8\n) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:\n    """Return H(M|H) - beta*H(M), normalized by log(number of motifs)."""\n    if assignments.ndim != 3 or assignments.shape[-1] < 2:\n        raise ValueError("assignments must have shape [batch, nodes, motifs>=2]")\n    probabilities = assignments.clamp_min(eps)\n    local_entropy_per_node = -(probabilities * probabilities.log()).sum(dim=-1)\n    h_local_raw = local_entropy_per_node.mean()\n    usage = assignments.mean(dim=(0, 1)).clamp_min(eps)\n    h_global_raw = -(usage * usage.log()).sum()\n    normalizer = math.log(assignments.shape[-1])\n    h_local_normalized = h_local_raw / normalizer\n    h_global_normalized = h_global_raw / normalizer\n    loss = h_local_normalized - beta * h_global_normalized\n    return loss, {\n        "H_local_raw": h_local_raw,\n        "H_local_normalized": h_local_normalized,\n        "H_global_raw": h_global_raw,\n        "H_global_normalized": h_global_normalized,\n        "L_MI": loss,\n        "prototype_utilization": usage,\n    }\n\n\ndef symmetric_js_divergence(\n    logits_a: torch.Tensor, logits_b: torch.Tensor, eps: float = 1e-8\n) -> torch.Tensor:\n    """Mean Jensen-Shannon divergence between two classifier distributions."""\n    p = F.softmax(logits_a, dim=-1)\n    q = F.softmax(logits_b, dim=-1)\n    m = 0.5 * (p + q)\n    log_m = m.clamp_min(eps).log()\n    kl_pm = (p * (F.log_softmax(logits_a, dim=-1) - log_m)).sum(dim=-1)\n    kl_qm = (q * (F.log_softmax(logits_b, dim=-1) - log_m)).sum(dim=-1)\n    return 0.5 * (kl_pm + kl_qm).mean()\n\n\ndef supervised_contrastive_loss(\n    embeddings: torch.Tensor,\n    labels: torch.Tensor,\n    temperature: float = 0.10,\n) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:\n    """Standard in-batch SupCon with safe exclusion of invalid anchors."""\n    if embeddings.ndim != 2:\n        raise ValueError("embeddings must have shape [batch, dimension]")\n    if labels.ndim != 1 or labels.shape[0] != embeddings.shape[0]:\n        raise ValueError("labels must have shape [batch]")\n    if temperature <= 0.0:\n        raise ValueError("temperature must be positive")\n\n    normalized = F.normalize(embeddings, dim=-1)\n    batch = embeddings.shape[0]\n    zero = embeddings.sum() * 0.0\n    if batch < 2:\n        return zero, {\n            "valid_supcon_anchor_fraction": zero.detach(),\n            "mean_positive_count": zero.detach(),\n        }\n\n    self_mask = torch.eye(batch, dtype=torch.bool, device=embeddings.device)\n    positive_mask = labels[:, None].eq(labels[None, :]) & ~self_mask\n    positive_count = positive_mask.sum(dim=1)\n    valid = positive_count > 0\n\n    similarity = normalized @ normalized.t() / temperature\n    similarity = similarity - similarity.max(dim=1, keepdim=True).values.detach()\n    denominator_logits = similarity.masked_fill(self_mask, -torch.inf)\n    log_prob = similarity - torch.logsumexp(denominator_logits, dim=1, keepdim=True)\n    per_anchor = -(\n        log_prob.masked_fill(~positive_mask, 0.0).sum(dim=1)\n        / positive_count.clamp_min(1)\n    )\n    loss = per_anchor[valid].mean() if bool(valid.any()) else zero\n    valid_fraction = valid.to(embeddings.dtype).mean()\n    mean_positive_count = (\n        positive_count[valid].to(embeddings.dtype).mean()\n        if bool(valid.any())\n        else zero.detach()\n    )\n    return loss, {\n        "valid_supcon_anchor_fraction": valid_fraction,\n        "mean_positive_count": mean_positive_count,\n    }\n', 'motif.py': '"""Differentiable multiscale spatial motif composition for MPG-FER v2.2."""\n\nfrom __future__ import annotations\n\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .losses import motif_mutual_information_loss\n\n\ndef scheduled_motif_temperature(\n    epoch: int,\n    tau_start: float = 0.70,\n    tau_final: float = 0.30,\n    anneal_end_epoch: int = 35,\n) -> float:\n    """Cosine-anneal motif temperature, then hold the registered floor."""\n    if epoch < 1:\n        raise ValueError("epoch must be >= 1")\n    if anneal_end_epoch < 2:\n        raise ValueError("anneal_end_epoch must be >= 2")\n    if not 0.0 < tau_final <= tau_start:\n        raise ValueError("temperature bounds must satisfy 0 < final <= start")\n    progress = min(max((epoch - 1) / (anneal_end_epoch - 1), 0.0), 1.0)\n    return tau_final + 0.5 * (tau_start - tau_final) * (\n        1.0 + math.cos(math.pi * progress)\n    )\n\n\ndef _reflect_index(index: int, size: int) -> int:\n    """Map an arbitrary integer to [0,size) using reflection without edge repeat."""\n    if size < 2:\n        return 0\n    period = 2 * (size - 1)\n    value = index % period\n    return value if value < size else period - value\n\n\ndef build_aligned_support_indices(\n    img_size: int = 48,\n    anchor_size: int = 12,\n    stride: int = 6,\n    scales: tuple[int, ...] = (8, 12, 16),\n) -> tuple[dict[int, torch.Tensor], torch.Tensor]:\n    """Build reflection-mapped supports sharing the 12x12 anchor centers."""\n    starts = list(range(0, img_size - anchor_size + 1, stride))\n    if len(starts) != 7:\n        raise ValueError("MPG-FER v2.2 requires a 7x7 anchor grid")\n    centers = []\n    supports: dict[int, list[list[int]]] = {scale: [] for scale in scales}\n    for top in starts:\n        for left in starts:\n            center_y = top + (anchor_size - 1) / 2.0\n            center_x = left + (anchor_size - 1) / 2.0\n            centers.append((center_x, center_y))\n            for scale in scales:\n                start_y = int(round(center_y - (scale - 1) / 2.0))\n                start_x = int(round(center_x - (scale - 1) / 2.0))\n                indices = [\n                    _reflect_index(row, img_size) * img_size\n                    + _reflect_index(column, img_size)\n                    for row in range(start_y, start_y + scale)\n                    for column in range(start_x, start_x + scale)\n                ]\n                supports[scale].append(indices)\n    return (\n        {scale: torch.tensor(values, dtype=torch.long) for scale, values in supports.items()},\n        torch.tensor(centers, dtype=torch.float32),\n    )\n\n\ndef aligned_logical_starts(\n    img_size: int = 48,\n    anchor_size: int = 12,\n    stride: int = 6,\n    scales: tuple[int, ...] = (8, 12, 16),\n) -> dict[int, torch.Tensor]:\n    """Return pre-reflection [top,left] starts for alignment audits."""\n    starts = list(range(0, img_size - anchor_size + 1, stride))\n    result: dict[int, list[tuple[int, int]]] = {scale: [] for scale in scales}\n    for top in starts:\n        for left in starts:\n            center_y = top + (anchor_size - 1) / 2.0\n            center_x = left + (anchor_size - 1) / 2.0\n            for scale in scales:\n                result[scale].append(\n                    (\n                        int(round(center_y - (scale - 1) / 2.0)),\n                        int(round(center_x - (scale - 1) / 2.0)),\n                    )\n                )\n    return {scale: torch.tensor(values, dtype=torch.long) for scale, values in result.items()}\n\n\nclass SpatialMotifComposer(nn.Module):\n    """Soft TYPE assignment and aligned 8/12/16 occurrence fusion into 49 nodes."""\n\n    def __init__(\n        self,\n        d_pixel: int = 96,\n        num_motifs: int = 48,\n        motif_window_sizes: tuple[int, int, int] = (8, 12, 16),\n        motif_stride: int = 6,\n        img_size: int = 48,\n        d_type: int = 32,\n        d_motif: int = 192,\n        tau_start: float = 0.70,\n        tau_final: float = 0.30,\n        tau_anneal_end_epoch: int = 35,\n        mi_beta: float = 1.0,\n        eps: float = 1e-6,\n    ) -> None:\n        super().__init__()\n        scheduled_motif_temperature(\n            1, tau_start=tau_start, tau_final=tau_final,\n            anneal_end_epoch=tau_anneal_end_epoch,\n        )\n        self.d_pixel = d_pixel\n        self.num_motifs = num_motifs\n        self.window_sizes = tuple(motif_window_sizes)\n        self.stride = motif_stride\n        self.img_size = img_size\n        self.d_type = d_type\n        self.d_motif = d_motif\n        self.tau_start = float(tau_start)\n        self.tau_final = float(tau_final)\n        self.tau_anneal_end_epoch = int(tau_anneal_end_epoch)\n        self.mi_beta = float(mi_beta)\n        self.eps = eps\n\n        self.prototypes = nn.Parameter(torch.randn(num_motifs, d_pixel) / math.sqrt(d_pixel))\n        self.assignment_query = nn.Linear(d_pixel, d_pixel, bias=False)\n        self.prototype_key = nn.Linear(d_pixel, d_pixel, bias=False)\n        self.register_buffer(\n            "current_tau", torch.tensor(self.tau_start, dtype=torch.float32)\n        )\n\n        self.scale_saliency = nn.ModuleDict(\n            {str(scale): nn.Linear(d_pixel, 1) for scale in self.window_sizes}\n        )\n        self.scale_confidence = nn.Parameter(torch.ones(len(self.window_sizes)))\n        self.type_proj = nn.Sequential(\n            nn.Linear(num_motifs, d_type), nn.LayerNorm(d_type), nn.GELU()\n        )\n        self.occurrence_proj = nn.Sequential(\n            nn.Linear(d_pixel + d_type + 5, d_motif),\n            nn.LayerNorm(d_motif),\n            nn.GELU(),\n        )\n        self.scale_gate = nn.Linear(d_motif, 1)\n\n        coords = torch.linspace(-1.0, 1.0, img_size)\n        grid_y, grid_x = torch.meshgrid(coords, coords, indexing="ij")\n        self.register_buffer("grid_x", grid_x.reshape(-1))\n        self.register_buffer("grid_y", grid_y.reshape(-1))\n        supports, centers = build_aligned_support_indices(\n            img_size=img_size,\n            anchor_size=12,\n            stride=motif_stride,\n            scales=self.window_sizes,\n        )\n        self.register_buffer("anchor_centers_pixel", centers)\n        for scale, indices in supports.items():\n            self.register_buffer(f"support_idx_{scale}", indices)\n\n    @property\n    def temperature(self) -> torch.Tensor:\n        return self.current_tau\n\n    @torch.no_grad()\n    def set_epoch_temperature(self, epoch: int) -> float:\n        value = scheduled_motif_temperature(\n            epoch,\n            tau_start=self.tau_start,\n            tau_final=self.tau_final,\n            anneal_end_epoch=self.tau_anneal_end_epoch,\n        )\n        self.current_tau.fill_(value)\n        return value\n\n    def _pool_scale(\n        self, h_pixel: torch.Tensor, assignments: torch.Tensor,\n        confidence: torch.Tensor, scale: int,\n    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:\n        batch = h_pixel.shape[0]\n        indices = getattr(self, f"support_idx_{scale}")\n        occurrences, support_size = indices.shape\n        gather_idx = indices.unsqueeze(0).expand(batch, -1, -1)\n        h_support = h_pixel[:, indices, :]\n        a_support = assignments[:, indices, :]\n        conf_support = confidence[:, indices, :]\n        x_support = self.grid_x[indices].view(1, occurrences, support_size, 1).expand(batch, -1, -1, -1)\n        y_support = self.grid_y[indices].view(1, occurrences, support_size, 1).expand(batch, -1, -1, -1)\n\n        saliency = self.scale_saliency[str(scale)](h_support)\n        scale_position = self.window_sizes.index(scale)\n        weights = F.softmax(saliency + self.scale_confidence[scale_position] * conf_support, dim=2)\n        what = (weights * h_support).sum(dim=2)\n        type_distribution = (weights * a_support).sum(dim=2)\n        motif_type = self.type_proj(type_distribution)\n        cx = (weights * x_support).sum(dim=2)\n        cy = (weights * y_support).sum(dim=2)\n        sx = torch.sqrt((weights * (x_support - cx.unsqueeze(2)).square()).sum(dim=2) + self.eps)\n        sy = torch.sqrt((weights * (y_support - cy.unsqueeze(2)).square()).sum(dim=2) + self.eps)\n        mass = (weights * conf_support).sum(dim=2)\n        where = torch.cat([cx, cy, sx, sy, mass], dim=-1)\n        candidate = self.occurrence_proj(torch.cat([what, motif_type, where], dim=-1))\n        centers = torch.cat([cx, cy], dim=-1)\n        return candidate, centers, weights.squeeze(-1), type_distribution\n\n    def forward(self, h_pixel: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, dict[str, torch.Tensor]]:\n        batch, nodes, dimension = h_pixel.shape\n        if nodes != self.img_size * self.img_size or dimension != self.d_pixel:\n            raise ValueError("Unexpected pixel embedding shape")\n\n        queries = F.normalize(self.assignment_query(h_pixel), dim=-1)\n        keys = F.normalize(self.prototype_key(self.prototypes), dim=-1)\n        assignments = F.softmax(queries @ keys.t() / self.temperature, dim=-1)\n        confidence = assignments.max(dim=-1, keepdim=True).values\n\n        candidates, centers, weights, type_distributions = [], [], {}, {}\n        for scale in self.window_sizes:\n            candidate, center, scale_weights, scale_types = self._pool_scale(\n                h_pixel, assignments, confidence, scale\n            )\n            candidates.append(candidate)\n            centers.append(center)\n            weights[str(scale)] = scale_weights\n            type_distributions[str(scale)] = scale_types\n        candidate_stack = torch.stack(candidates, dim=2)  # [B,49,3,192]\n        center_stack = torch.stack(centers, dim=2)  # [B,49,3,2]\n        alpha = F.softmax(self.scale_gate(candidate_stack).squeeze(-1), dim=-1)\n        h_motif = (alpha.unsqueeze(-1) * candidate_stack).sum(dim=2)\n        fused_centers = (alpha.unsqueeze(-1) * center_stack).sum(dim=2)\n\n        p_norm = F.normalize(self.prototypes, dim=-1)\n        prototype_cosine = p_norm @ p_norm.t()\n        offdiag_mask = ~torch.eye(self.num_motifs, dtype=torch.bool, device=h_pixel.device)\n        offdiag = prototype_cosine[offdiag_mask]\n        diversity = F.relu(offdiag).square().mean()\n        mi_loss, mi = motif_mutual_information_loss(assignments, beta=self.mi_beta)\n        utilization = mi["prototype_utilization"]\n        top2 = assignments.topk(2, dim=-1).values\n        mean_entropy = mi["H_local_raw"]\n        diagnostics = {\n            "loss_diversity": diversity,\n            "loss_mi": mi_loss,\n            "tau": self.temperature,\n            "H_local_raw": mi["H_local_raw"],\n            "H_local_normalized": mi["H_local_normalized"],\n            "H_global_raw": mi["H_global_raw"],\n            "H_global_normalized": mi["H_global_normalized"],\n            "L_MI": mi["L_MI"],\n            "mean_entropy": mean_entropy,\n            "effective_motif_count": mean_entropy.exp(),\n            "min_utilization": utilization.min(),\n            "max_utilization": utilization.max(),\n            "std_utilization": utilization.std(unbiased=False),\n            "mean_top1_probability": top2[..., 0].mean(),\n            "mean_top2_probability": top2[..., 1].mean(),\n            "mean_top1_top2_margin": (top2[..., 0] - top2[..., 1]).mean(),\n            "mean_offdiag_prototype_cosine": offdiag.mean(),\n            "learned_centers_x": fused_centers[..., 0],\n            "learned_centers_y": fused_centers[..., 1],\n            "scale_weights": alpha,\n            "mean_alpha_8": alpha[..., 0].mean(),\n            "mean_alpha_12": alpha[..., 1].mean(),\n            "mean_alpha_16": alpha[..., 2].mean(),\n            "std_alpha_8": alpha[..., 0].std(unbiased=False),\n            "std_alpha_12": alpha[..., 1].std(unbiased=False),\n            "std_alpha_16": alpha[..., 2].std(unbiased=False),\n            "occurrence_weights_8": weights["8"],\n            "occurrence_weights_12": weights["12"],\n            "occurrence_weights_16": weights["16"],\n            "type_distributions": sum(\n                alpha[..., index].unsqueeze(-1) * type_distributions[str(scale)]\n                for index, scale in enumerate(self.window_sizes)\n            ),\n        }\n        return h_motif, assignments, diagnostics\n', 'model.py': '"""Pure-GNN MPG-FER v2.2 model: pixel graph, motifs, and motif graph."""\n\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .config import MPGConfig\nfrom .features import PixelFeatureExtractor\nfrom .graph import PixelGraphTopology\nfrom .motif import SpatialMotifComposer\n\n\nclass DropPath(nn.Module):\n    """Per-sample stochastic depth applied to residual branches, never nodes."""\n\n    def __init__(self, drop_probability: float = 0.0) -> None:\n        super().__init__()\n        if not 0.0 <= drop_probability < 1.0:\n            raise ValueError("drop_probability must be in [0,1)")\n        self.drop_probability = float(drop_probability)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        if not self.training or self.drop_probability == 0.0:\n            return x\n        keep = 1.0 - self.drop_probability\n        shape = (x.shape[0],) + (1,) * (x.ndim - 1)\n        mask = torch.empty(shape, dtype=x.dtype, device=x.device).bernoulli_(keep)\n        return x * mask / keep\n\n\ndef masked_neighbor_softmax(scores: torch.Tensor, neighbor_mask: torch.Tensor) -> torch.Tensor:\n    mask = neighbor_mask.unsqueeze(0).unsqueeze(2).unsqueeze(3)\n    weights = F.softmax(scores.masked_fill(~mask, torch.finfo(scores.dtype).min), dim=-1)\n    return weights.masked_fill(~mask, 0.0)\n\n\ndef compute_motif_geometry(cx: torch.Tensor, cy: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:\n    dx = cx.unsqueeze(1) - cx.unsqueeze(2)\n    dy = cy.unsqueeze(1) - cy.unsqueeze(2)\n    dist_sq = dx.square() + dy.square()\n    dist = torch.sqrt(dist_sq + eps)\n    nodes = cx.shape[1]\n    self_mask = torch.eye(nodes, device=cx.device, dtype=torch.bool).unsqueeze(0)\n    dist = dist.masked_fill(self_mask, 0.0)\n    sin_theta = (dy / (dist + eps)).masked_fill(self_mask, 0.0)\n    cos_theta = (dx / (dist + eps)).masked_fill(self_mask, 0.0)\n    return torch.stack([dx, dy, dist, dist_sq, sin_theta, cos_theta], dim=-1)\n\n\nclass EdgeAwarePixelGNNLayer(nn.Module):\n    """8-neighbor attention with K/V projected once per node before gather."""\n\n    def __init__(\n        self, d_pixel: int = 96, edge_dim: int = 5, num_heads: int = 4,\n        dropout: float = 0.15, drop_path: float = 0.0,\n    ) -> None:\n        super().__init__()\n        if d_pixel % num_heads:\n            raise ValueError("d_pixel must be divisible by num_heads")\n        self.d_pixel = d_pixel\n        self.num_heads = num_heads\n        self.head_dim = d_pixel // num_heads\n        self.norm1 = nn.LayerNorm(d_pixel)\n        self.q_proj = nn.Linear(d_pixel, d_pixel)\n        self.k_proj = nn.Linear(d_pixel, d_pixel)\n        self.v_proj = nn.Linear(d_pixel, d_pixel)\n        self.edge_bias = nn.Linear(edge_dim, num_heads)\n        self.edge_val = nn.Linear(edge_dim, d_pixel)\n        self.out_proj = nn.Linear(d_pixel, d_pixel)\n        self.attn_dropout = nn.Dropout(dropout)\n        self.drop_path1 = DropPath(drop_path)\n        self.norm2 = nn.LayerNorm(d_pixel)\n        self.ffn = nn.Sequential(\n            nn.Linear(d_pixel, 2 * d_pixel), nn.GELU(), nn.Dropout(dropout),\n            nn.Linear(2 * d_pixel, d_pixel), nn.Dropout(dropout),\n        )\n        self.drop_path2 = DropPath(drop_path)\n\n    def forward(\n        self, h: torch.Tensor, neighbor_idx: torch.Tensor,\n        neighbor_mask: torch.Tensor, edge_feats: torch.Tensor,\n    ) -> torch.Tensor:\n        batch, nodes, dimension = h.shape\n        heads, head_dim = self.num_heads, self.head_dim\n        normalized = self.norm1(h)\n        q = self.q_proj(normalized).reshape(batch, nodes, heads, head_dim).unsqueeze(3)\n        k_all = self.k_proj(normalized)\n        v_all = self.v_proj(normalized)\n        k = k_all[:, neighbor_idx, :].reshape(batch, nodes, 8, heads, head_dim).permute(0, 1, 3, 2, 4)\n        v = (v_all[:, neighbor_idx, :] + self.edge_val(edge_feats)).reshape(\n            batch, nodes, 8, heads, head_dim\n        ).permute(0, 1, 3, 2, 4)\n        scores = torch.matmul(q, k.transpose(-1, -2)) / (head_dim**0.5)\n        scores = scores + self.edge_bias(edge_feats).permute(0, 1, 3, 2).unsqueeze(3)\n        attention = self.attn_dropout(masked_neighbor_softmax(scores, neighbor_mask))\n        message = torch.matmul(attention, v).squeeze(3).reshape(batch, nodes, dimension)\n        h = h + self.drop_path1(self.out_proj(message))\n        return h + self.drop_path2(self.ffn(self.norm2(h)))\n\n\nclass GeometryAwareMotifTransformerBlock(nn.Module):\n    def __init__(\n        self, d_motif: int = 192, geom_dim: int = 6, num_heads: int = 6,\n        dropout: float = 0.15, drop_path: float = 0.0, topk: int = 48,\n    ) -> None:\n        super().__init__()\n        if d_motif % num_heads:\n            raise ValueError("d_motif must be divisible by num_heads")\n        if isinstance(topk, bool) or not isinstance(topk, int) or topk < 1:\n            raise ValueError("topk must be a positive integer")\n        self.d_motif = d_motif\n        self.num_heads = num_heads\n        self.head_dim = d_motif // num_heads\n        self.topk = topk\n        self.norm1 = nn.LayerNorm(d_motif)\n        self.q_proj = nn.Linear(d_motif, d_motif)\n        self.k_proj = nn.Linear(d_motif, d_motif)\n        self.v_proj = nn.Linear(d_motif, d_motif)\n        self.geom_proj = nn.Linear(geom_dim, num_heads)\n        self.out_proj = nn.Linear(d_motif, d_motif)\n        self.attn_dropout = nn.Dropout(dropout)\n        self.drop_path1 = DropPath(drop_path)\n        self.norm2 = nn.LayerNorm(d_motif)\n        self.ffn = nn.Sequential(\n            nn.Linear(d_motif, 2 * d_motif), nn.GELU(), nn.Dropout(dropout),\n            nn.Linear(2 * d_motif, d_motif), nn.Dropout(dropout),\n        )\n        self.drop_path2 = DropPath(drop_path)\n\n    def forward(\n        self, h_motif: torch.Tensor, geom_edges: torch.Tensor, return_diagnostics: bool = False\n    ) -> torch.Tensor | tuple[torch.Tensor, dict[str, torch.Tensor]]:\n        batch, nodes, dimension = h_motif.shape\n        normalized = self.norm1(h_motif)\n        reshape = lambda value: value.reshape(batch, nodes, self.num_heads, self.head_dim).permute(0, 2, 1, 3)\n        q, k, v = reshape(self.q_proj(normalized)), reshape(self.k_proj(normalized)), reshape(self.v_proj(normalized))\n        scores = q @ k.transpose(-1, -2) / (self.head_dim**0.5)\n        scores = scores + self.geom_proj(geom_edges).permute(0, 3, 1, 2)\n        if self.topk > nodes - 1:\n            raise ValueError(\n                f"topk={self.topk} exceeds the {nodes - 1} non-self keys"\n            )\n        self_mask = torch.eye(\n            nodes, device=h_motif.device, dtype=torch.bool\n        ).view(1, 1, nodes, nodes)\n        self_mask = self_mask.expand(batch, self.num_heads, nodes, nodes)\n        masked_scores = scores.masked_fill(self_mask, torch.finfo(scores.dtype).min)\n        if self.topk < nodes - 1:\n            topk_result = torch.topk(\n                masked_scores, k=self.topk, dim=-1\n            )\n            topk_indices = topk_result.indices\n            selected_mask = torch.zeros_like(masked_scores, dtype=torch.bool)\n            selected_mask.scatter_(-1, topk_indices, True)\n            cutoff = topk_result.values[..., -1:]\n            greater_count = (masked_scores > cutoff).sum(dim=-1)\n            equal_count = (masked_scores == cutoff).sum(dim=-1)\n            boundary_tie_count = (\n                equal_count > (self.topk - greater_count)\n            ).sum()\n            masked_scores = masked_scores.masked_fill(\n                ~selected_mask, torch.finfo(scores.dtype).min\n            )\n        else:\n            selected_mask = ~self_mask\n            boundary_tie_count = torch.zeros(\n                (), dtype=torch.long, device=h_motif.device\n            )\n        attention_pre_dropout = F.softmax(masked_scores, dim=-1).masked_fill(\n            ~selected_mask, 0.0\n        )\n        attention = self.attn_dropout(attention_pre_dropout)\n        message = (attention @ v).permute(0, 2, 1, 3).reshape(batch, nodes, dimension)\n        h_motif = h_motif + self.drop_path1(self.out_proj(message))\n        h_motif = h_motif + self.drop_path2(self.ffn(self.norm2(h_motif)))\n\n        if not return_diagnostics:\n            return h_motif\n\n        diagnostic_attention = attention_pre_dropout.detach()\n        p = diagnostic_attention.clamp(min=1e-12)\n        entropy = -(diagnostic_attention * p.log()).sum(dim=-1).mean()\n        top1 = diagnostic_attention.max(dim=-1).values.mean()\n        layer_diag = {\n            "topk": torch.tensor(self.topk, device=h_motif.device),\n            "selected_mask": selected_mask.detach(),\n            "attention_pre_dropout": diagnostic_attention,\n            "entropy": entropy,\n            "top1_mass": top1,\n            "boundary_tie_count": boundary_tie_count.detach(),\n        }\n        return h_motif, layer_diag\n\n\ndef _linear_rates(depth: int, maximum: float) -> list[float]:\n    return torch.linspace(0.0, maximum, depth).tolist() if depth > 1 else [maximum]\n\n\nclass MPGFER(nn.Module):\n    """MPG-FER v2.2; no convolutional or dense-image transformer backbone."""\n\n    def __init__(self, config: MPGConfig | None = None) -> None:\n        super().__init__()\n        self.config = config or MPGConfig()\n        cfg = self.config\n        self.pixel_extractor = PixelFeatureExtractor(img_size=cfg.img_size)\n        self.pixel_topology = PixelGraphTopology(img_size=cfg.img_size)\n        self.pixel_proj = nn.Sequential(\n            nn.Linear(cfg.raw_pixel_dim, cfg.d_pixel), nn.LayerNorm(cfg.d_pixel), nn.GELU()\n        )\n        self.pixel_gnn = nn.ModuleList([\n            EdgeAwarePixelGNNLayer(\n                cfg.d_pixel, cfg.pixel_edge_dim, cfg.num_pixel_heads,\n                cfg.pixel_dropout, drop_path=rate,\n            )\n            for rate in _linear_rates(cfg.num_pixel_gnn_layers, cfg.pixel_drop_path_max)\n        ])\n        self.pixel_attn_pool = nn.Linear(cfg.d_pixel, 1)\n        self.pixel_readout_proj = nn.Sequential(\n            nn.Linear(cfg.d_pixel * 3, cfg.d_pixel_readout),\n            nn.LayerNorm(cfg.d_pixel_readout), nn.GELU(),\n        )\n        self.aux_pixel_head = nn.Linear(cfg.d_pixel_readout, cfg.num_classes)\n        self.motif_composer = SpatialMotifComposer(\n            d_pixel=cfg.d_pixel, num_motifs=cfg.num_motifs,\n            motif_window_sizes=cfg.motif_window_sizes, motif_stride=cfg.motif_stride,\n            img_size=cfg.img_size, d_type=cfg.d_type, d_motif=cfg.d_motif,\n            tau_start=cfg.tau_start, tau_final=cfg.tau_final,\n            tau_anneal_end_epoch=cfg.tau_anneal_end_epoch,\n            mi_beta=cfg.mi_beta,\n        )\n        self.motif_gnn = nn.ModuleList([\n            GeometryAwareMotifTransformerBlock(\n                cfg.d_motif, cfg.motif_geom_dim, cfg.num_motif_heads,\n                cfg.motif_dropout, drop_path=rate,\n                topk=cfg.motif_topk_schedule[i],\n            )\n            for i, rate in enumerate(_linear_rates(cfg.num_motif_layers, cfg.motif_drop_path_max))\n        ])\n        self.motif_attn_pool = nn.Linear(cfg.d_motif, 1)\n        self.motif_readout_proj = nn.Sequential(\n            nn.Linear(cfg.d_motif * 3, cfg.d_motif_readout),\n            nn.LayerNorm(cfg.d_motif_readout), nn.GELU(),\n        )\n        self.aux_motif_head = nn.Linear(cfg.d_motif_readout, cfg.num_classes)\n        self.supcon_head = nn.Sequential(\n            nn.Linear(cfg.d_classifier_in, cfg.supcon_dim),\n            nn.LayerNorm(cfg.supcon_dim),\n        )\n        self.classifier = nn.Sequential(\n            nn.Linear(cfg.d_classifier_in, cfg.d_classifier_hidden),\n            nn.LayerNorm(cfg.d_classifier_hidden), nn.GELU(),\n            nn.Dropout(cfg.classifier_dropout),\n            nn.Linear(cfg.d_classifier_hidden, cfg.num_classes),\n        )\n\n    def set_epoch_temperature(self, epoch: int) -> float:\n        """Set the deterministic serialized motif temperature for an epoch."""\n        return self.motif_composer.set_epoch_temperature(epoch)\n\n    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:\n        batch = x.shape[0]\n        h = self.pixel_proj(self.pixel_extractor(x))\n        intensities = x.reshape(batch, self.config.num_pixels, 1)\n        edges = self.pixel_topology.compute_edge_features(intensities)\n        for layer in self.pixel_gnn:\n            h = layer(h, self.pixel_topology.neighbor_idx, self.pixel_topology.neighbor_mask, edges)\n        p_mean, p_max = h.mean(dim=1), h.max(dim=1).values\n        p_attention = (F.softmax(self.pixel_attn_pool(h), dim=1) * h).sum(dim=1)\n        pixel_readout = self.pixel_readout_proj(torch.cat([p_mean, p_max, p_attention], dim=-1))\n        pixel_logits = self.aux_pixel_head(pixel_readout)\n\n        h_motif, assignments, diagnostics = self.motif_composer(h)\n        geometry = compute_motif_geometry(\n            diagnostics["learned_centers_x"], diagnostics["learned_centers_y"]\n        )\n        routing_diagnostics = {}\n        for l_idx, layer in enumerate(self.motif_gnn):\n            h_motif, layer_diag = layer(h_motif, geometry, return_diagnostics=True)\n            routing_diagnostics[f"motif_l{l_idx+1}_entropy"] = layer_diag["entropy"]\n            routing_diagnostics[f"motif_l{l_idx+1}_top1_mass"] = layer_diag["top1_mass"]\n            routing_diagnostics[f"motif_l{l_idx+1}_boundary_tie_count"] = (\n                layer_diag["boundary_tie_count"]\n            )\n        m_mean, m_max = h_motif.mean(dim=1), h_motif.max(dim=1).values\n        m_attention = (F.softmax(self.motif_attn_pool(h_motif), dim=1) * h_motif).sum(dim=1)\n        motif_readout = self.motif_readout_proj(torch.cat([m_mean, m_max, m_attention], dim=-1))\n        motif_logits = self.aux_motif_head(motif_readout)\n        fusion = torch.cat([pixel_readout, motif_readout], dim=-1)\n        supcon_embeddings = F.normalize(self.supcon_head(fusion), dim=-1)\n        logits = self.classifier(fusion)\n        outputs = {\n            "final_logits": logits,\n            "pixel_logits": pixel_logits,\n            "motif_logits": motif_logits,\n            "h_pixel_readout": pixel_readout,\n            "h_motif_readout": motif_readout,\n            "fusion_representation": fusion,\n            "supcon_embeddings": supcon_embeddings,\n            "motif_assignments": assignments,\n            "motif_geometry": geometry,\n            **diagnostics,\n            **routing_diagnostics,\n        }\n        return logits, outputs\n', 'ema.py': '"""Complete model-state exponential moving average."""\n\nfrom __future__ import annotations\n\nimport copy\nimport torch\nimport torch.nn as nn\n\n\nclass ModelEMA:\n    """Track parameters and buffers and expose deterministic evaluation state."""\n\n    def __init__(self, model: nn.Module, decay: float = 0.999) -> None:\n        if not 0.0 <= decay < 1.0:\n            raise ValueError("EMA decay must be in [0, 1)")\n        self.decay = float(decay)\n        self.num_updates = 0\n        self.module = copy.deepcopy(model).eval()\n        self.module.requires_grad_(False)\n\n    @torch.no_grad()\n    def update(self, model: nn.Module) -> None:\n        source = model.state_dict()\n        target = self.module.state_dict()\n        parameter_names = set(dict(model.named_parameters()))\n        if source.keys() != target.keys():\n            raise RuntimeError("EMA/model state keys differ")\n        for name, ema_value in target.items():\n            value = source[name].detach()\n            if name in parameter_names and torch.is_floating_point(ema_value):\n                ema_value.mul_(self.decay).add_(value, alpha=1.0 - self.decay)\n            else:\n                # Scheduled state such as current_tau is authoritative state,\n                # not a quantity that EMA is allowed to numerically average.\n                ema_value.copy_(value)\n        self.num_updates += 1\n\n    def state_dict(self) -> dict:\n        return {\n            "decay": self.decay,\n            "num_updates": self.num_updates,\n            "model_state_dict": self.module.state_dict(),\n        }\n\n    def load_state_dict(self, state: dict) -> None:\n        self.decay = float(state["decay"])\n        self.num_updates = int(state["num_updates"])\n        self.module.load_state_dict(state["model_state_dict"], strict=True)\n', 'checkpoint.py': '"""Atomic, hash-verified, full-state resume bundles for MPG-FER v2.2."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nfrom typing import Any\n\nimport numpy as np\nimport torch\n\nfrom .config import MPGConfig\n\n\ndef sha256_file(path: str | Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef canonical_config(config: MPGConfig, include_runtime_safe: bool = False) -> dict:\n    data = asdict(config)\n    safe = set(config.runtime_safe_resume_fields) | {"runtime_safe_resume_fields"}\n    return data if include_runtime_safe else {k: v for k, v in data.items() if k not in safe}\n\n\ndef config_hash(config: MPGConfig) -> str:\n    payload = json.dumps(canonical_config(config), sort_keys=True, separators=(",", ":"))\n    return hashlib.sha256(payload.encode("utf-8")).hexdigest()\n\n\ndef capture_rng_state(loader_generator: torch.Generator) -> dict:\n    return {\n        "python": random.getstate(),\n        "numpy": np.random.get_state(),\n        "torch_cpu": torch.get_rng_state(),\n        "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n        "dataloader_generator": loader_generator.get_state(),\n    }\n\n\ndef restore_rng_state(state: dict, loader_generator: torch.Generator) -> None:\n    random.setstate(state["python"])\n    np.random.set_state(state["numpy"])\n    torch.set_rng_state(state["torch_cpu"])\n    if torch.cuda.is_available() and state["torch_cuda"]:\n        torch.cuda.set_rng_state_all(state["torch_cuda"])\n    loader_generator.set_state(state["dataloader_generator"])\n\n\ndef build_resume_bundle(\n    *, config: MPGConfig, run_id: str, source_hash: str, completed_epoch: int,\n    global_optimizer_step: int, model: torch.nn.Module, ema: Any,\n    optimizer: torch.optim.Optimizer, scheduler: Any, scaler: Any,\n    best_comparator_state: dict | None, best_epoch: int | None,\n    best_metrics: dict | None, early_stop_counter: int, history: list[dict],\n    loader_generator: torch.Generator, consistency_state: dict,\n    sampler_state: dict | None = None,\n    augmentation_state: dict | None = None,\n) -> dict:\n    temperature_state = None\n    if hasattr(model, "motif_composer"):\n        temperature_state = {\n            "epoch": completed_epoch,\n            "current_tau": float(model.motif_composer.temperature),\n        }\n    return {\n        "resume_schema_version": config.resume_schema_version,\n        "run_id": run_id,\n        "source_hash": source_hash,\n        "config_hash": config_hash(config),\n        "scientific_config": canonical_config(config),\n        "completed_epoch": completed_epoch,\n        "next_epoch": completed_epoch + 1,\n        "global_optimizer_step": global_optimizer_step,\n        "model_state_dict": model.state_dict(),\n        "ema_state_dict": ema.state_dict(),\n        "optimizer_state_dict": optimizer.state_dict(),\n        "scheduler_state_dict": scheduler.state_dict(),\n        "grad_scaler_state_dict": scaler.state_dict() if scaler is not None else {},\n        "best_comparator_state": best_comparator_state,\n        "best_epoch": best_epoch,\n        "best_metrics": best_metrics,\n        "early_stop_counter": early_stop_counter,\n        "early_stop_monitor_state": {\n            "start_epoch": config.early_stop_monitor_start_epoch,\n            "active": completed_epoch >= config.early_stop_monitor_start_epoch,\n            "patience": early_stop_counter,\n        },\n        "temperature_state": temperature_state,\n        "history": history,\n        "rng_state": capture_rng_state(loader_generator),\n        "consistency_state": consistency_state,\n        "sampler_state": sampler_state,\n        "augmentation_state": augmentation_state,\n    }\n\n\ndef atomic_save_resume(bundle: dict, output_dir: str | Path, status: str = "TRAINING") -> tuple[Path, str]:\n    output = Path(output_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    target = output / "resume_latest.pt"\n    temporary = output / "resume_latest.tmp"\n    with temporary.open("wb") as handle:\n        torch.save(bundle, handle)\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(temporary, target)\n    digest = sha256_file(target)\n    metadata = {\n        "run_id": bundle["run_id"],\n        "epoch": bundle["completed_epoch"],\n        "next_epoch": bundle["next_epoch"],\n        "checkpoint_sha256": digest,\n        "source_hash": bundle["source_hash"],\n        "config_hash": bundle["config_hash"],\n        "timestamp": datetime.now(timezone.utc).isoformat(),\n        "status": status,\n    }\n    meta_tmp = output / "resume_latest.json.tmp"\n    meta_target = output / "resume_latest.json"\n    with meta_tmp.open("w", encoding="utf-8") as handle:\n        json.dump(metadata, handle, indent=2)\n        handle.write("\\n")\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(meta_tmp, meta_target)\n    return target, digest\n\n\ndef _atomic_json(path: Path, payload: dict) -> None:\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", encoding="utf-8") as handle:\n        json.dump(payload, handle, indent=2)\n        handle.write("\\n")\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(temporary, path)\n\n\ndef save_periodic_snapshot(latest: Path, epoch: int, interval: int, keep: int) -> Path | None:\n    if interval <= 0 or epoch % interval:\n        return None\n    snapshot = latest.with_name(f"resume_epoch_{epoch:03d}.pt")\n    if not snapshot.exists():\n        temporary = snapshot.with_suffix(".tmp")\n        with latest.open("rb") as source, temporary.open("wb") as destination:\n            shutil.copyfileobj(source, destination, length=1024 * 1024)\n            destination.flush()\n            os.fsync(destination.fileno())\n        os.replace(temporary, snapshot)\n        bundle = torch.load(snapshot, map_location="cpu", weights_only=False)\n        _atomic_json(\n            snapshot.with_suffix(".json"),\n            {\n                "checkpoint_sha256": sha256_file(snapshot),\n                "resume_schema_version": bundle["resume_schema_version"],\n                "run_id": bundle["run_id"],\n                "source_hash": bundle["source_hash"],\n                "config_hash": bundle["config_hash"],\n                "completed_epoch": bundle["completed_epoch"],\n                "next_epoch": bundle["next_epoch"],\n                "status": "IMMUTABLE_FALLBACK",\n            },\n        )\n    snapshots = sorted(latest.parent.glob("resume_epoch_*.pt"))\n    for old in snapshots[:-max(keep, 2)]:\n        old.unlink()\n        old.with_suffix(".json").unlink(missing_ok=True)\n    return snapshot\n\n\ndef find_latest_valid_snapshot(\n    directory: str | Path,\n    expected_identity: dict | None = None,\n) -> tuple[Path, str] | None:\n    """Report the newest independently hash-valid immutable fallback."""\n    root = Path(directory)\n    for metadata_path in sorted(root.glob("resume_epoch_*.json"), reverse=True):\n        try:\n            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n            if expected_identity is not None and any(\n                expected_identity.get(key) is not None\n                and metadata.get(key) != expected_identity[key]\n                for key in (\n                    "resume_schema_version", "run_id", "source_hash", "config_hash"\n                )\n            ):\n                continue\n            checkpoint = metadata_path.with_suffix(".pt")\n            expected = metadata["checkpoint_sha256"]\n            if checkpoint.is_file() and sha256_file(checkpoint) == expected:\n                return checkpoint, expected\n        except (OSError, KeyError, ValueError, json.JSONDecodeError):\n            continue\n    return None\n\n\ndef load_resume_bundle(\n    path: str | Path, *, expected_sha256: str | None, config: MPGConfig,\n    run_id: str | None, source_hash: str,\n) -> dict:\n    checkpoint = Path(path)\n    if expected_sha256 and sha256_file(checkpoint) != expected_sha256:\n        raise RuntimeError("Resume SHA-256 mismatch")\n    bundle = torch.load(checkpoint, map_location="cpu", weights_only=False)\n    if bundle.get("resume_schema_version") != config.resume_schema_version:\n        raise RuntimeError("Resume schema mismatch")\n    if run_id is not None and bundle.get("run_id") != run_id:\n        raise RuntimeError("Resume run_id mismatch")\n    if bundle.get("source_hash") != source_hash:\n        raise RuntimeError("Resume source hash mismatch")\n    if bundle.get("config_hash") != config_hash(config):\n        raise RuntimeError("Resume scientific/architecture config mismatch")\n    monitor = bundle.get("early_stop_monitor_state")\n    if monitor != {\n        "start_epoch": config.early_stop_monitor_start_epoch,\n        "active": bundle["completed_epoch"] >= config.early_stop_monitor_start_epoch,\n        "patience": bundle["early_stop_counter"],\n    }:\n        raise RuntimeError("Resume early-stop monitor state mismatch")\n    temperature = bundle.get("temperature_state")\n    if temperature is not None:\n        state_tau = bundle["model_state_dict"].get("motif_composer.current_tau")\n        if (\n            temperature.get("epoch") != bundle["completed_epoch"]\n            or state_tau is None\n            or not torch.isclose(\n                torch.as_tensor(state_tau).cpu(),\n                torch.tensor(float(temperature["current_tau"])),\n                rtol=0.0,\n                atol=1e-7,\n            )\n        ):\n            raise RuntimeError("Resume scheduled temperature state mismatch")\n    return bundle\n\n\ndef restore_training_state(\n    bundle: dict, *, model: torch.nn.Module, ema: Any,\n    optimizer: torch.optim.Optimizer, scheduler: Any, scaler: Any,\n    loader_generator: torch.Generator,\n    sampler: Any | None = None,\n    dataset: Any | None = None,\n) -> None:\n    model.load_state_dict(bundle["model_state_dict"], strict=True)\n    ema.load_state_dict(bundle["ema_state_dict"])\n    optimizer.load_state_dict(bundle["optimizer_state_dict"])\n    scheduler.load_state_dict(bundle["scheduler_state_dict"])\n    if scaler is not None and bundle["grad_scaler_state_dict"]:\n        scaler.load_state_dict(bundle["grad_scaler_state_dict"])\n    restore_rng_state(bundle["rng_state"], loader_generator)\n    if sampler is not None and bundle.get("sampler_state") is not None:\n        sampler.load_state_dict(bundle["sampler_state"])\n    if dataset is not None and bundle.get("augmentation_state") is not None:\n        dataset.load_state_dict(bundle["augmentation_state"])\n', 'data.py': '"""FER2013 loading with split isolation and resume-stable augmentation."""\n\nfrom __future__ import annotations\n\nimport csv\nfrom pathlib import Path\nfrom typing import Tuple\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, Dataset, Sampler\nimport torchvision.transforms.functional as TF\n\n\nEXPECTED_TRAIN_ROWS = 28709\nEXPECTED_PUBLIC_ROWS = 3589\nEXPECTED_PRIVATE_ROWS = 3589\nROLE_TO_BASENAME = {\n    "train": "train.csv",\n    "val": "val.csv",\n    "test": "test.csv",\n}\nROLE_TO_EXPECTED_ROWS = {\n    "train": EXPECTED_TRAIN_ROWS,\n    "val": EXPECTED_PUBLIC_ROWS,\n    "test": EXPECTED_PRIVATE_ROWS,\n}\n\n\ndef augmentation_seed(base_seed: int, epoch: int, sample_index: int) -> int:\n    """Stable sample-local seed, independent of worker process RNG history."""\n    return int(base_seed) + int(epoch) * 1_000_003 + int(sample_index) * 97_409\n\n\nclass EpochRandomSampler(Sampler[int]):\n    """Derive each epoch\'s permutation solely from base seed and epoch."""\n\n    def __init__(self, data_source: Dataset, seed: int = 42) -> None:\n        self.data_source = data_source\n        self.seed = int(seed)\n        self.epoch = 0\n\n    def set_epoch(self, epoch: int) -> None:\n        self.epoch = int(epoch)\n\n    def __iter__(self):\n        generator = torch.Generator().manual_seed(\n            self.seed + self.epoch * 2_000_033\n        )\n        return iter(torch.randperm(len(self.data_source), generator=generator).tolist())\n\n    def __len__(self) -> int:\n        return len(self.data_source)\n\n    def state_dict(self) -> dict:\n        return {"scheme": "base_seed_epoch_v1", "seed": self.seed, "epoch": self.epoch}\n\n    def load_state_dict(self, state: dict) -> None:\n        if state.get("scheme") != "base_seed_epoch_v1" or int(state["seed"]) != self.seed:\n            raise RuntimeError("Training sampler state is incompatible")\n        self.epoch = int(state["epoch"])\n\n\ndef validate_split_path(csv_path: str | Path, role: str) -> Path:\n    """Validate a split by explicit role and basename, never by row count alone."""\n    normalized_role = role.lower()\n    if normalized_role not in ROLE_TO_BASENAME:\n        raise ValueError(f"Unknown FER2013 split role: {role!r}")\n    path = Path(csv_path)\n    expected = ROLE_TO_BASENAME[normalized_role]\n    if path.name.lower() != expected:\n        raise ValueError(\n            f"Split role {normalized_role!r} requires basename {expected!r}; "\n            f"refusing suspicious path {path}"\n        )\n    if not path.is_file():\n        raise FileNotFoundError(f"FER2013 {normalized_role} CSV not found: {path}")\n    return path\n\n\ndef validate_split_paths(\n    train_csv: str | Path,\n    val_csv: str | Path,\n    test_csv: str | Path | None = None,\n) -> tuple[Path, Path, Path | None]:\n    """Fail closed on role swaps, aliases, and duplicate split paths."""\n    train_path = validate_split_path(train_csv, "train")\n    val_path = validate_split_path(val_csv, "val")\n    test_path = validate_split_path(test_csv, "test") if test_csv is not None else None\n    resolved = [p.resolve() for p in (train_path, val_path, test_path) if p is not None]\n    if len(set(resolved)) != len(resolved):\n        raise ValueError("Train, PublicTest, and PrivateTest paths must be distinct")\n    return train_path, val_path, test_path\n\n\ndef inspect_split_file(\n    csv_path: str | Path,\n    role: str,\n    validate_content: bool,\n) -> dict:\n    """Validate split metadata and, when authorized, every labeled image row."""\n    path = validate_split_path(csv_path, role)\n    normalized_role = role.lower()\n    rows = 0\n    with path.open("r", encoding="utf-8", newline="") as handle:\n        reader = csv.reader(handle)\n        try:\n            header = [column.strip().lower() for column in next(reader)]\n        except StopIteration as exc:\n            raise ValueError(f"Empty FER2013 CSV: {path}") from exc\n        if "pixels" not in header or "emotion" not in header:\n            raise ValueError(f"CSV requires emotion and pixels columns: {path}")\n        pixel_index = header.index("pixels")\n        emotion_index = header.index("emotion")\n        for row in reader:\n            if not row:\n                continue\n            rows += 1\n            if validate_content:\n                pixels = np.fromstring(row[pixel_index], sep=" ", dtype=np.int16)\n                if len(pixels) != 2304:\n                    raise ValueError(\n                        f"Malformed pixel row with {len(pixels)} pixels in {path}"\n                    )\n                if np.any((pixels < 0) | (pixels > 255)):\n                    raise ValueError(f"Pixel outside [0, 255] in {path}")\n                label = int(row[emotion_index])\n                if not 0 <= label <= 6:\n                    raise ValueError(f"Invalid label {label} in {path}")\n    expected_rows = ROLE_TO_EXPECTED_ROWS[normalized_role]\n    if rows != expected_rows:\n        raise ValueError(\n            f"{normalized_role} requires exactly {expected_rows} rows, got {rows}"\n        )\n    return {\n        "role": normalized_role,\n        "path": str(path.resolve()),\n        "rows": rows,\n        "expected_rows": expected_rows,\n        "content_validated": validate_content,\n    }\n\n\ndef validate_dataset_gate(\n    train_csv: str | Path,\n    val_csv: str | Path,\n    test_csv: str | Path,\n) -> dict[str, dict]:\n    """Phase-A gate: inspect Train/Public content, Private metadata only."""\n    train_path, val_path, test_path = validate_split_paths(\n        train_csv, val_csv, test_csv\n    )\n    assert test_path is not None\n    return {\n        "train": inspect_split_file(train_path, "train", validate_content=True),\n        "public": inspect_split_file(val_path, "val", validate_content=True),\n        # PrivateTest content stays locked until the frozen-checkpoint boundary.\n        "private": inspect_split_file(test_path, "test", validate_content=False),\n    }\n\n\nclass FER2013Dataset(Dataset):\n    """FER2013 CSV Dataset returning normalized [1, 48, 48] images and labels."""\n\n    def __init__(\n        self,\n        csv_path: str | Path,\n        split: str = "train",\n        augment: bool = False,\n        seed: int = 42,\n    ) -> None:\n        super().__init__()\n        aliases = {"public": "val", "private": "test"}\n        self.split = aliases.get(split.lower(), split.lower())\n        if self.split not in ROLE_TO_BASENAME:\n            raise ValueError(f"Unknown FER2013 split role: {split!r}")\n        self.augment = augment and (self.split == "train")\n        self.seed = int(seed)\n        self.epoch = 0\n\n        p = validate_split_path(csv_path, self.split)\n\n        images_list = []\n        labels_list = []\n\n        with open(p, "r", encoding="utf-8", newline="") as f:\n            reader = csv.reader(f)\n            header = [c.strip().lower() for c in next(reader)]\n            if "pixels" not in header:\n                raise ValueError(f"CSV missing \'pixels\' column: {p}")\n            pix_idx = header.index("pixels")\n            if "emotion" not in header:\n                raise ValueError(f"CSV missing \'emotion\' column: {p}")\n            emo_idx = header.index("emotion")\n\n            for row in reader:\n                if not row:\n                    continue\n                pix = np.fromstring(row[pix_idx], sep=" ", dtype=np.int16)\n                if len(pix) != 2304:\n                    raise ValueError(f"Malformed pixel row with {len(pix)} pixels in {p}")\n                if np.any((pix < 0) | (pix > 255)):\n                    raise ValueError(f"Pixel outside [0, 255] in {p}")\n                images_list.append(pix.astype(np.uint8).reshape(48, 48))\n\n                label = int(row[emo_idx])\n                if not 0 <= label <= 6:\n                    raise ValueError(f"Invalid label {label} in {p}")\n                labels_list.append(label)\n\n        if not images_list:\n            raise ValueError(f"FER2013 CSV contains no data rows: {p}")\n        self.images = np.stack(images_list, axis=0)  # [N, 48, 48] uint8\n        self.labels = np.array(labels_list, dtype=np.int64)\n\n        if self.split == "train" and len(self.images) != EXPECTED_TRAIN_ROWS:\n            warnings_msg = f"Expected {EXPECTED_TRAIN_ROWS} rows for train, got {len(self.images)}"\n            print(f"[Warning] {warnings_msg}")\n        elif self.split in ("val", "public") and len(self.images) != EXPECTED_PUBLIC_ROWS:\n            print(f"[Warning] Expected {EXPECTED_PUBLIC_ROWS} rows for val, got {len(self.images)}")\n        elif self.split in ("test", "private") and len(self.images) != EXPECTED_PRIVATE_ROWS:\n            print(f"[Warning] Expected {EXPECTED_PRIVATE_ROWS} rows for test, got {len(self.images)}")\n\n    def __len__(self) -> int:\n        return len(self.images)\n\n    def set_epoch(self, epoch: int) -> None:\n        """Select deterministic, sample-local augmentation for an epoch."""\n        self.epoch = int(epoch)\n\n    def state_dict(self) -> dict:\n        return {\n            "scheme": "sample_local_seed_v1",\n            "seed": self.seed,\n            "epoch": self.epoch,\n        }\n\n    def load_state_dict(self, state: dict) -> None:\n        if state.get("scheme") != "sample_local_seed_v1" or int(state["seed"]) != self.seed:\n            raise RuntimeError("Training augmentation state is incompatible")\n        self.epoch = int(state["epoch"])\n\n    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:\n        img_np = self.images[idx]\n        img_tensor = torch.from_numpy(img_np).float().unsqueeze(0) / 255.0  # [1, 48, 48] in [0, 1]\n        label = int(self.labels[idx])\n\n        if self.augment:\n            generator = torch.Generator().manual_seed(\n                augmentation_seed(self.seed, self.epoch, int(idx))\n            )\n            # Conservative FER-safe augmentations:\n            # 1. Random horizontal flip (p = 0.5)\n            if torch.rand(1, generator=generator).item() > 0.5:\n                img_tensor = TF.hflip(img_tensor)\n\n            # 2. Random translation (crop/shift by up to 2 pixels)\n            if torch.rand(1, generator=generator).item() > 0.5:\n                shift_y = int(torch.randint(-2, 3, (1,), generator=generator).item())\n                shift_x = int(torch.randint(-2, 3, (1,), generator=generator).item())\n                img_tensor = TF.affine(\n                    img_tensor,\n                    angle=0.0,\n                    translate=[shift_x, shift_y],\n                    scale=1.0,\n                    shear=[0.0, 0.0],\n                    fill=0.0,\n                )\n\n            # 3. Very mild intensity scaling: [0.95, 1.05]\n            if torch.rand(1, generator=generator).item() > 0.5:\n                scale = 0.95 + 0.10 * torch.rand(1, generator=generator).item()\n                img_tensor = torch.clamp(img_tensor * scale, 0.0, 1.0)\n\n        return img_tensor, label\n\n\ndef create_training_dataloaders(\n    train_csv: str | Path,\n    val_csv: str | Path,\n    batch_size: int = 16,\n    num_workers: int = 2,\n    seed: int = 42,\n    generator: torch.Generator | None = None,\n) -> dict[str, DataLoader]:\n    """Create only Train and PublicTest loaders; PrivateTest stays unopened."""\n    train_path, val_path, _ = validate_split_paths(train_csv, val_csv)\n    train_ds = FER2013Dataset(train_path, split="train", augment=True, seed=seed)\n    val_ds = FER2013Dataset(val_path, split="val", augment=False)\n    if generator is None:\n        generator = torch.Generator().manual_seed(seed)\n    train_sampler = EpochRandomSampler(train_ds, seed=seed)\n\n    loaders = {\n        "train": DataLoader(\n            train_ds,\n            batch_size=batch_size,\n            shuffle=False,\n            sampler=train_sampler,\n            num_workers=num_workers,\n            pin_memory=True,\n            drop_last=False,\n            generator=generator,\n        ),\n        "val": DataLoader(\n            val_ds,\n            batch_size=batch_size,\n            shuffle=False,\n            num_workers=num_workers,\n            pin_memory=True,\n            drop_last=False,\n        ),\n    }\n\n    return loaders\n\n\ndef create_private_dataloader(\n    test_csv: str | Path,\n    batch_size: int = 16,\n    num_workers: int = 2,\n) -> DataLoader:\n    """Construct the PrivateTest loader only after checkpoint freeze."""\n    test_path = validate_split_path(test_csv, "test")\n    test_ds = FER2013Dataset(test_path, split="test", augment=False)\n    return DataLoader(\n        test_ds,\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=num_workers,\n        pin_memory=True,\n        drop_last=False,\n    )\n', 'evaluate.py': '"""Evaluation utilities with single-pass raw and horizontal-flip TTA metrics."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nfrom sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\nimport torchvision.transforms.functional as TF\n\n\ndef _classification_metrics(\n    targets: list[int], predictions: list[int], total_loss: float\n) -> dict:\n    y_true = np.asarray(targets, dtype=np.int64)\n    y_pred = np.asarray(predictions, dtype=np.int64)\n    if len(y_true) == 0:\n        raise ValueError("Cannot evaluate an empty dataloader")\n    return {\n        "loss": float(total_loss / len(y_true)),\n        "accuracy": float(np.mean(y_true == y_pred)),\n        "macro_f1": float(\n            f1_score(y_true, y_pred, average="macro", zero_division=0)\n        ),\n        "per_class_f1": [\n            float(x)\n            for x in f1_score(\n                y_true, y_pred, average=None, labels=np.arange(7), zero_division=0\n            ).tolist()\n        ],\n        "precision": [\n            float(x)\n            for x in precision_score(\n                y_true, y_pred, average=None, labels=np.arange(7), zero_division=0\n            ).tolist()\n        ],\n        "recall": [\n            float(x)\n            for x in recall_score(\n                y_true, y_pred, average=None, labels=np.arange(7), zero_division=0\n            ).tolist()\n        ],\n        "confusion_matrix": confusion_matrix(\n            y_true, y_pred, labels=np.arange(7)\n        ).tolist(),\n        "support": [int((y_true == label).sum()) for label in range(7)],\n    }\n\n\n@torch.no_grad()\ndef evaluate_raw_and_tta(\n    model: nn.Module,\n    dataloader: DataLoader,\n    device: str | torch.device,\n    use_amp: bool = True,\n) -> dict[str, dict]:\n    """Compute raw and logit-averaged flip-TTA metrics in one loader traversal.\n\n    The image is horizontally flipped before the model extracts relational\n    features. PrivateTest callers therefore make exactly one physical pass over\n    the loader while obtaining both reporting views from the frozen weights.\n    """\n    model.eval()\n    raw_preds: list[int] = []\n    tta_preds: list[int] = []\n    targets_all: list[int] = []\n    raw_loss_total = 0.0\n    tta_loss_total = 0.0\n    criterion = nn.CrossEntropyLoss()\n    amp_enabled = use_amp and torch.cuda.is_available()\n\n    for images, targets in dataloader:\n        images = images.to(device)\n        targets = targets.to(device)\n        with torch.amp.autocast("cuda", enabled=amp_enabled):\n            raw_logits, _ = model(images)\n            flipped_logits, _ = model(TF.hflip(images))\n            tta_logits = 0.5 * (raw_logits + flipped_logits)\n            raw_loss = criterion(raw_logits, targets)\n            tta_loss = criterion(tta_logits, targets)\n\n        batch_size = len(targets)\n        raw_loss_total += raw_loss.item() * batch_size\n        tta_loss_total += tta_loss.item() * batch_size\n        raw_preds.extend(torch.argmax(raw_logits, dim=-1).cpu().tolist())\n        tta_preds.extend(torch.argmax(tta_logits, dim=-1).cpu().tolist())\n        targets_all.extend(targets.cpu().tolist())\n\n    return {\n        "raw": _classification_metrics(targets_all, raw_preds, raw_loss_total),\n        "tta": _classification_metrics(targets_all, tta_preds, tta_loss_total),\n    }\n\n\n@torch.no_grad()\ndef evaluate_model(\n    model: nn.Module,\n    dataloader: DataLoader,\n    device: str | torch.device,\n    use_tta: bool = False,\n    use_amp: bool = True,\n) -> dict:\n    """Compatibility wrapper selecting one view from the single-pass evaluator."""\n    both = evaluate_raw_and_tta(model, dataloader, device, use_amp=use_amp)\n    selected = dict(both["tta" if use_tta else "raw"])\n    selected["use_tta"] = use_tta\n    return selected\n', 'utils.py': '"""Utility functions for training, logging, checkpointing, and reproducibility."""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom pathlib import Path\nimport random\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\n\ndef set_seed(seed: int = 42) -> None:\n    """Ensure full determinism across Python, NumPy, and PyTorch."""\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False\n\n\ndef save_checkpoint(\n    checkpoint_path: str | Path,\n    model: nn.Module,\n    optimizer: torch.optim.Optimizer,\n    epoch: int,\n    val_metrics: dict,\n    config_dict: dict,\n    extra_state: dict | None = None,\n) -> None:\n    """Save an atomic, reproducible experiment checkpoint."""\n    p = Path(checkpoint_path)\n    p.parent.mkdir(parents=True, exist_ok=True)\n\n    state = {\n        "epoch": epoch,\n        "model_state_dict": model.state_dict(),\n        "optimizer_state_dict": optimizer.state_dict(),\n        "val_metrics": val_metrics,\n        "config": config_dict,\n    }\n    if extra_state is not None:\n        state.update(extra_state)\n\n    torch.save(state, p)\n\n\ndef load_checkpoint(\n    checkpoint_path: str | Path,\n    model: nn.Module,\n    optimizer: torch.optim.Optimizer | None = None,\n) -> dict:\n    """Load model and optimizer state from checkpoint."""\n    p = Path(checkpoint_path)\n    if not p.is_file():\n        raise FileNotFoundError(f"Checkpoint not found: {p}")\n    checkpoint = torch.load(p, map_location="cpu")\n    model.load_state_dict(checkpoint["model_state_dict"])\n    if optimizer is not None and "optimizer_state_dict" in checkpoint:\n        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])\n    return checkpoint\n', 'train.py': '"""EMA-selected training and exact epoch-boundary continuation for MPG-FER v2.2."""\n\nfrom __future__ import annotations\n\nimport csv\nfrom dataclasses import asdict\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport time\nimport uuid\n\nimport torch\nimport torch.nn as nn\nfrom torch.optim import AdamW\nimport torchvision.transforms.functional as TF\n\nfrom .checkpoint import (\n    atomic_save_resume,\n    build_resume_bundle,\n    load_resume_bundle,\n    restore_training_state,\n    save_periodic_snapshot,\n    sha256_file,\n)\nfrom .config import MPGConfig\nfrom .data import (\n    FER2013Dataset,\n    create_private_dataloader,\n    create_training_dataloaders,\n    validate_split_path,\n)\nfrom .ema import ModelEMA\nfrom .evaluate import evaluate_raw_and_tta\nfrom .losses import supervised_contrastive_loss, symmetric_js_divergence\nfrom .model import MPGFER\nfrom .utils import set_seed\n\n\nMOTIF_DIAGNOSTICS = (\n    "tau", "H_local_raw", "H_local_normalized", "H_global_raw",\n    "H_global_normalized", "L_MI", "mean_entropy", "effective_motif_count",\n    "min_utilization", "max_utilization", "std_utilization",\n    "mean_top1_probability", "mean_top2_probability", "mean_top1_top2_margin",\n    "mean_offdiag_prototype_cosine", "mean_alpha_8", "mean_alpha_12",\n    "mean_alpha_16", "std_alpha_8", "std_alpha_12", "std_alpha_16",\n)\n\n\nclass WarmupCosineScheduler:\n    """Warm up, decay to an independent horizon, then hold the LR floor."""\n\n    def __init__(\n        self, optimizer, base_lr: float, warmup_epochs: int,\n        lr_decay_end_epoch: int, max_epochs: int, eta_min: float,\n    ) -> None:\n        if not 0 < warmup_epochs < lr_decay_end_epoch <= max_epochs:\n            raise ValueError(\n                "require 0 < warmup_epochs < lr_decay_end_epoch <= max_epochs"\n            )\n        self.optimizer = optimizer\n        self.base_lr = float(base_lr)\n        self.warmup_epochs = int(warmup_epochs)\n        self.lr_decay_end_epoch = int(lr_decay_end_epoch)\n        self.max_epochs = int(max_epochs)\n        self.eta_min = float(eta_min)\n        self.last_epoch = 0\n\n    def lr_for_epoch(self, epoch: int) -> float:\n        if not 1 <= epoch <= self.max_epochs:\n            raise ValueError(f"epoch must be in [1,{self.max_epochs}]")\n        if epoch <= self.warmup_epochs:\n            return self.base_lr * epoch / self.warmup_epochs\n        progress = (epoch - self.warmup_epochs) / (\n            self.lr_decay_end_epoch - self.warmup_epochs\n        )\n        progress = min(max(progress, 0.0), 1.0)\n        return self.eta_min + 0.5 * (self.base_lr - self.eta_min) * (1.0 + math.cos(math.pi * progress))\n\n    def step(self, epoch: int) -> float:\n        lr = self.lr_for_epoch(epoch)\n        for group in self.optimizer.param_groups:\n            group["lr"] = lr\n        self.last_epoch = epoch\n        return lr\n\n    def state_dict(self) -> dict:\n        return {\n            "base_lr": self.base_lr, "warmup_epochs": self.warmup_epochs,\n            "lr_decay_end_epoch": self.lr_decay_end_epoch,\n            "max_epochs": self.max_epochs, "eta_min": self.eta_min,\n            "last_epoch": self.last_epoch,\n        }\n\n    def load_state_dict(self, state: dict) -> None:\n        expected = {\n            "base_lr": self.base_lr,\n            "warmup_epochs": self.warmup_epochs,\n            "lr_decay_end_epoch": self.lr_decay_end_epoch,\n            "max_epochs": self.max_epochs,\n            "eta_min": self.eta_min,\n        }\n        for name, value in expected.items():\n            if state[name] != value:\n                raise RuntimeError(f"Scheduler {name} mismatch")\n        self.last_epoch = int(state["last_epoch"])\n\n\ndef source_tree_hash(package_dir: str | Path | None = None) -> str:\n    root = Path(package_dir) if package_dir is not None else Path(__file__).resolve().parent\n    digest = hashlib.sha256()\n    for path in sorted(root.glob("*.py")):\n        digest.update(path.name.encode("utf-8"))\n        digest.update(path.read_bytes())\n    return digest.hexdigest()\n\n\ndef is_better_checkpoint(candidate: dict, incumbent: dict | None) -> bool:\n    if incumbent is None:\n        return True\n    return (candidate["accuracy"], candidate["macro_f1"], -candidate["loss"]) > (\n        incumbent["accuracy"], incumbent["macro_f1"], -incumbent["loss"]\n    )\n\n\ndef update_early_stop_patience(\n    epoch: int, improved: bool, patience_counter: int, config: MPGConfig,\n) -> int:\n    """Select checkpoints always, but consume patience only from epoch 85."""\n    if improved or epoch < config.early_stop_monitor_start_epoch:\n        return 0\n    return patience_counter + 1\n\n\ndef should_early_stop(epoch: int, patience_counter: int, config: MPGConfig) -> bool:\n    return (\n        epoch >= config.early_stop_monitor_start_epoch\n        and patience_counter >= config.early_stop_patience\n    )\n\n\ndef should_end_segment(\n    elapsed_seconds: float,\n    estimated_next_epoch_seconds: float,\n    config: MPGConfig,\n) -> bool:\n    safety_margin = config.segment_safety_margin_minutes * 60.0\n    return (\n        elapsed_seconds + estimated_next_epoch_seconds + safety_margin\n        >= config.segment_soft_limit_hours * 3600.0\n    )\n\n\ndef consistency_selected(seed: int, epoch: int, group_index: int, probability: float) -> bool:\n    """Stateless decision, stable across process and resume boundaries."""\n    value = random.Random(seed + epoch * 1_000_003 + group_index * 9_176).random()\n    return value < probability\n\n\ndef compute_training_loss(\n    logits: torch.Tensor, outputs: dict[str, torch.Tensor], targets: torch.Tensor,\n    criterion: nn.Module, config: MPGConfig, flipped_logits: torch.Tensor | None = None,\n) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:\n    weighted = {\n        "ce_final": criterion(logits, targets),\n        "weighted_ce_motif": config.aux_motif_weight * criterion(outputs["motif_logits"], targets),\n        "weighted_ce_pixel": config.aux_pixel_weight * criterion(outputs["pixel_logits"], targets),\n        "weighted_diversity": config.lambda_div * outputs["loss_diversity"],\n        "weighted_mi": config.lambda_mi * outputs["loss_mi"],\n    }\n    raw_consistency = logits.new_zeros(())\n    if flipped_logits is not None:\n        raw_consistency = symmetric_js_divergence(logits, flipped_logits)\n    weighted["weighted_consistency"] = config.lambda_consistency * raw_consistency\n    raw_supcon, supcon_stats = supervised_contrastive_loss(\n        outputs["supcon_embeddings"], targets, config.supcon_temperature\n    )\n    weighted["supcon_loss_weighted"] = config.lambda_supcon * raw_supcon\n    total = sum(weighted.values())\n    components = {\n        **weighted,\n        "consistency_loss_raw": raw_consistency,\n        "supcon_loss_raw": raw_supcon,\n        **supcon_stats,\n    }\n    return total, components\n\n\ndef train_one_epoch(\n    model: nn.Module, dataloader: torch.utils.data.DataLoader,\n    optimizer: torch.optim.Optimizer, device: str | torch.device,\n    scaler: torch.amp.GradScaler | None, config: MPGConfig, criterion: nn.Module,\n    ema: ModelEMA | None = None, epoch: int = 1, global_optimizer_step: int = 0,\n) -> tuple[dict, int]:\n    model.train()\n    if hasattr(model, "set_epoch_temperature"):\n        model.set_epoch_temperature(epoch)\n    if ema is not None and hasattr(ema.module, "set_epoch_temperature"):\n        ema.module.set_epoch_temperature(epoch)\n    optimizer.zero_grad(set_to_none=True)\n    component_names = (\n        "ce_final", "weighted_ce_motif", "weighted_ce_pixel",\n        "weighted_diversity", "weighted_mi", "consistency_loss_raw",\n        "weighted_consistency", "supcon_loss_raw", "supcon_loss_weighted",\n        "valid_supcon_anchor_fraction", "mean_positive_count",\n    )\n    totals = {name: 0.0 for name in ("train_loss", *component_names, *MOTIF_DIAGNOSTICS)}\n    correct = samples = 0\n    accum_steps = config.gradient_accumulation_steps\n    use_amp = config.use_amp and torch.cuda.is_available()\n    num_batches = len(dataloader)\n    selected_groups: list[int] = []\n\n    for step, (images, targets) in enumerate(dataloader):\n        images, targets = images.to(device), targets.to(device)\n        group_index = step // accum_steps\n        group_start = group_index * accum_steps\n        group_size = min(accum_steps, num_batches - group_start)\n        use_consistency = consistency_selected(\n            config.seed, epoch, group_index, config.consistency_probability\n        )\n        if use_consistency and group_index not in selected_groups:\n            selected_groups.append(group_index)\n        with torch.amp.autocast("cuda", enabled=use_amp):\n            logits, outputs = model(images)\n            flipped_logits = model(TF.hflip(images))[0] if use_consistency else None\n            loss, components = compute_training_loss(\n                logits, outputs, targets, criterion, config, flipped_logits\n            )\n            backward_loss = loss / group_size\n        if scaler is not None:\n            scaler.scale(backward_loss).backward()\n        else:\n            backward_loss.backward()\n\n        end_group = (step + 1) % accum_steps == 0 or step + 1 == num_batches\n        if end_group:\n            if scaler is not None:\n                scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)\n            successful = True\n            if scaler is not None:\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                successful = scaler.get_scale() >= old_scale\n            else:\n                optimizer.step()\n            if successful:\n                global_optimizer_step += 1\n                if ema is not None:\n                    ema.update(model)\n            optimizer.zero_grad(set_to_none=True)\n\n        batch_size = len(targets)\n        totals["train_loss"] += float(loss.detach()) * batch_size\n        for name, value in components.items():\n            totals[name] += float(value.detach()) * batch_size\n        for name in MOTIF_DIAGNOSTICS:\n            totals[name] += float(outputs[name].detach()) * batch_size\n        correct += int((logits.argmax(dim=-1) == targets).sum())\n        samples += batch_size\n\n    stats = {name: value / samples for name, value in totals.items()}\n    stats["train_accuracy"] = correct / samples\n    stats["consistency_groups"] = selected_groups\n    stats["consistency_group_fraction"] = len(selected_groups) / math.ceil(num_batches / accum_steps)\n    return stats, global_optimizer_step\n\n\ndef run_micro_overfit_preflight(\n    train_csv: str | Path, config: MPGConfig | None = None,\n    device: str | torch.device | None = None, raise_on_failure: bool = True,\n) -> dict:\n    config = config or MPGConfig()\n    path = validate_split_path(train_csv, "train")\n    device = torch.device(device or (config.device if torch.cuda.is_available() else "cpu"))\n    set_seed(config.seed)\n    dataset = FER2013Dataset(path, split="train", augment=False)\n    images = torch.stack([dataset[i][0] for i in range(config.micro_overfit_samples)]).to(device)\n    targets = torch.tensor([dataset[i][1] for i in range(config.micro_overfit_samples)], device=device)\n    model = MPGFER(config).to(device)\n    model.set_epoch_temperature(1)\n    optimizer = AdamW(model.parameters(), lr=config.micro_overfit_learning_rate, weight_decay=0.0)\n    criterion = nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)\n    accuracy, final_loss, steps = 0.0, float("nan"), 0\n    for steps in range(1, config.micro_overfit_max_steps + 1):\n        model.train(); optimizer.zero_grad(set_to_none=True)\n        logits, outputs = model(images)\n        loss, _ = compute_training_loss(logits, outputs, targets, criterion, config)\n        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip); optimizer.step()\n        if steps % 5 == 0 or steps == config.micro_overfit_max_steps:\n            model.eval()\n            with torch.no_grad():\n                logits, outputs = model(images)\n                final_loss = float(compute_training_loss(logits, outputs, targets, criterion, config)[0])\n                accuracy = float((logits.argmax(dim=-1) == targets).float().mean())\n            if accuracy >= config.micro_overfit_target:\n                break\n    result = {"samples": config.micro_overfit_samples, "accuracy": accuracy,\n              "target": config.micro_overfit_target, "steps": steps,\n              "final_loss": final_loss, "passed": accuracy >= config.micro_overfit_target}\n    if not result["passed"] and raise_on_failure:\n        raise RuntimeError(f"16-example micro-overfit preflight failed: {result}")\n    return result\n\n\ndef run_bounded_runtime_preflight(\n    train_csv: str | Path, config: MPGConfig | None = None,\n    device: str | torch.device | None = None,\n) -> dict:\n    config = config or MPGConfig()\n    path = validate_split_path(train_csv, "train")\n    device = torch.device(device or config.device)\n    if device.type != "cuda" or not torch.cuda.is_available():\n        raise RuntimeError("CUDA is required for the bounded runtime preflight")\n    set_seed(config.seed)\n    dataset = FER2013Dataset(path, split="train", augment=False)\n    images = torch.stack([dataset[i][0] for i in range(config.batch_size)]).to(device)\n    targets = torch.tensor([dataset[i][1] for i in range(config.batch_size)], device=device)\n    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(device)\n    model = MPGFER(config).to(device).train()\n    model.set_epoch_temperature(1)\n    ema = ModelEMA(model, config.ema_decay)\n    optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)\n    scaler = torch.amp.GradScaler("cuda")\n    criterion = nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)\n    optimizer.zero_grad(set_to_none=True)\n    torch.cuda.synchronize(device)\n    step_started = time.perf_counter()\n    with torch.amp.autocast("cuda", enabled=True):\n        logits, outputs = model(images)\n        flipped_logits, _ = model(TF.hflip(images))\n        loss, components = compute_training_loss(\n            logits, outputs, targets, criterion, config, flipped_logits\n        )\n    if not torch.isfinite(loss) or not all(\n        torch.isfinite(value).all() for value in components.values()\n    ):\n        raise FloatingPointError("Non-finite bounded preflight loss")\n    scaler.scale(loss).backward(); scaler.unscale_(optimizer)\n    gradient_names = {\n        "assignment_query": model.motif_composer.assignment_query.weight,\n        "prototype_key": model.motif_composer.prototype_key.weight,\n        "prototypes": model.motif_composer.prototypes,\n        "scale_gate": model.motif_composer.scale_gate.weight,\n        "pixel_projection": model.pixel_proj[0].weight,\n        "pixel_readout": model.pixel_readout_proj[0].weight,\n        "motif_readout": model.motif_readout_proj[0].weight,\n        "supcon_projection": model.supcon_head[0].weight,\n        "classifier": model.classifier[-1].weight,\n    }\n    gradients = {name: bool(p.grad is not None and torch.isfinite(p.grad).all() and torch.any(p.grad != 0)) for name, p in gradient_names.items()}\n    if not all(gradients.values()):\n        raise FloatingPointError(f"Invalid gradients: {gradients}")\n    torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)\n    old_scale = scaler.get_scale()\n    scaler.step(optimizer); scaler.update()\n    optimizer_step_succeeded = scaler.get_scale() >= old_scale\n    if optimizer_step_succeeded:\n        ema.update(model)\n    torch.cuda.synchronize(device)\n    step_time_seconds = time.perf_counter() - step_started\n    return {\n        "physical_batch": config.batch_size,\n        "gradient_accumulation": config.gradient_accumulation_steps,\n        "amp_enabled": True,\n        "worst_case_consistency_forward": True,\n        "supcon_enabled": True,\n        "loss": float(loss.detach()),\n        "loss_components": {name: float(value.detach()) for name, value in components.items()},\n        "gradient_audits": gradients,\n        "optimizer_step_succeeded": optimizer_step_succeeded,\n        "ema_updates": ema.num_updates,\n        "step_time_seconds": step_time_seconds,\n        "peak_allocated_mib": torch.cuda.max_memory_allocated(device) / 2**20,\n        "peak_reserved_mib": torch.cuda.max_memory_reserved(device) / 2**20,\n        "passed": True,\n    }\n\n\ndef _write_history(history: list[dict], output: Path) -> None:\n    json_path = output / "history.json"\n    json_temporary = output / "history.json.tmp"\n    with json_temporary.open("w", encoding="utf-8") as handle:\n        json.dump(history, handle, indent=2)\n        handle.write("\\n")\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(json_temporary, json_path)\n    if not history:\n        return\n    rows = []\n    for entry in history:\n        flat = {k: v for k, v in entry.items() if not isinstance(v, (dict, list))}\n        for view in ("val_raw", "val_tta"):\n            for metric in ("loss", "accuracy", "macro_f1"):\n                flat[f"{view}_{metric}"] = entry[view][metric]\n        rows.append(flat)\n    csv_path = output / "history.csv"\n    csv_temporary = output / "history.csv.tmp"\n    with csv_temporary.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(csv_temporary, csv_path)\n\n\ndef _atomic_json_document(path: Path, payload: dict) -> None:\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", encoding="utf-8") as handle:\n        json.dump(payload, handle, indent=2)\n        handle.write("\\n")\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(temporary, path)\n\n\ndef _save_best_ema(path: Path, ema: ModelEMA, epoch: int, metrics: dict, config: MPGConfig, source_hash: str) -> None:\n    temporary = path.with_suffix(".tmp")\n    scheduled_tau = (\n        float(ema.module.motif_composer.temperature)\n        if hasattr(ema.module, "motif_composer")\n        else None\n    )\n    with temporary.open("wb") as handle:\n        torch.save({\n            "checkpoint_type": "EMA_INFERENCE_ONLY", "weights_type": "EMA", "epoch": epoch,\n            "model_state_dict": ema.module.state_dict(), "ema_state_dict": ema.state_dict(),\n            "scheduled_tau": scheduled_tau,\n            "val_metrics": metrics, "config": asdict(config), "source_hash": source_hash,\n            "selection_metric": "EMA PublicTest flip-TTA accuracy",\n            "selection_tiebreak": ["higher EMA TTA macro-F1", "lower EMA TTA loss"],\n        }, handle)\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(temporary, path)\n\n\ndef _segment_manifest(\n    output: Path, *, run_id: str, segment_number: int, start_epoch: int,\n    end_epoch: int, next_epoch: int, wallclock: float, resume_sha: str,\n    best_epoch: int | None, best_metrics: dict | None, status: str,\n) -> dict:\n    manifest = {\n        "run_id": run_id, "segment_number": segment_number,\n        "start_epoch": start_epoch, "end_epoch": end_epoch, "next_epoch": next_epoch,\n        "wallclock_seconds": wallclock, "resume_sha256": resume_sha,\n        "best_epoch": best_epoch,\n        "best_public_tta_accuracy": None if best_metrics is None else best_metrics["accuracy"],\n        "status": status,\n    }\n    _atomic_json_document(output / "segment_manifest.json", manifest)\n    return manifest\n\n\ndef run_training(\n    train_csv: str | Path, val_csv: str | Path, output_dir: str | Path,\n    preflight_result: dict, config: MPGConfig | None = None,\n    resume_path: str | Path | None = None, resume_sha256: str | None = None,\n) -> dict:\n    """Train/select with EMA PublicTest metrics; never opens PrivateTest."""\n    config = config or MPGConfig()\n    fresh_gate = (\n        preflight_result.get("passed")\n        and preflight_result.get("samples") == config.micro_overfit_samples\n    )\n    resume_gate = resume_path is not None and preflight_result.get(\n        "resume_compatibility_passed"\n    )\n    if not (fresh_gate or resume_gate):\n        raise RuntimeError(\n            "Training refused: valid real-16 fresh gate or verified resume gate required"\n        )\n    set_seed(config.seed)\n    device = torch.device(config.device if torch.cuda.is_available() else "cpu")\n    output = Path(output_dir); output.mkdir(parents=True, exist_ok=True)\n    source_hash = source_tree_hash()\n    generator = torch.Generator().manual_seed(config.seed)\n    loaders = create_training_dataloaders(\n        train_csv, val_csv, config.batch_size, config.num_workers,\n        seed=config.seed, generator=generator,\n    )\n    model = MPGFER(config).to(device)\n    optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)\n    scheduler = WarmupCosineScheduler(\n        optimizer, config.learning_rate, config.warmup_epochs,\n        config.lr_decay_end_epoch, config.max_epochs, config.min_learning_rate,\n    )\n    scaler = torch.amp.GradScaler("cuda") if config.use_amp and torch.cuda.is_available() else None\n    ema = ModelEMA(model, config.ema_decay)\n    criterion = nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)\n\n    run_id = config.run_id or str(uuid.uuid4())\n    start_epoch, global_step = 1, 0\n    best_metrics = best_comparator = None\n    best_epoch = None\n    patience = 0\n    history: list[dict] = []\n    resumed_from = None\n    if resume_path is not None:\n        bundle = load_resume_bundle(\n            resume_path, expected_sha256=resume_sha256, config=config,\n            run_id=config.run_id, source_hash=source_hash,\n        )\n        restore_training_state(\n            bundle, model=model, ema=ema, optimizer=optimizer,\n            scheduler=scheduler, scaler=scaler, loader_generator=generator,\n            sampler=loaders["train"].sampler,\n            dataset=loaders["train"].dataset,\n        )\n        run_id = bundle["run_id"]\n        start_epoch = int(bundle["next_epoch"])\n        global_step = int(bundle["global_optimizer_step"])\n        best_comparator = bundle["best_comparator_state"]\n        best_metrics = bundle["best_metrics"]\n        best_epoch = bundle["best_epoch"]\n        patience = int(bundle["early_stop_counter"])\n        history = list(bundle["history"])\n        resumed_from = str(Path(resume_path).resolve())\n        print("RESUMING EXISTING MPG-FER v2.2 RUN")\n        print(f"run_id={run_id}")\n        print(f"completed epoch={bundle[\'completed_epoch\']}")\n        print(f"next epoch={start_epoch}")\n        print(f"restored LR={optimizer.param_groups[0][\'lr\']:.10f}")\n        print(f"restored optimizer step={global_step}")\n        print(f"restored best epoch={best_epoch}")\n        print(f"restored patience={patience}")\n        print(f"restored tau={float(model.motif_composer.temperature):.10f}")\n        print(f"restored EMA updates={ema.num_updates}")\n        print(\n            "restored GradScaler scale="\n            + ("none" if scaler is None else f"{scaler.get_scale():.10f}")\n        )\n        print(\n            "early-stop monitor active="\n            f"{start_epoch >= config.early_stop_monitor_start_epoch}"\n        )\n\n    (output / "config.json").write_text(json.dumps(asdict(config), indent=2), encoding="utf-8")\n    segment_started = time.monotonic()\n    segment_start_epoch = start_epoch\n    status = "TRAINING_COMPLETED"\n    resume_digest = ""\n    end_epoch = start_epoch - 1\n    checkpoint_path = output / "best_val_acc.pt"\n    print(f"Starting MPG-FER v2.2 training on device: {device}; run_id={run_id}")\n\n    for epoch in range(start_epoch, config.max_epochs + 1):\n        loaders["train"].dataset.set_epoch(epoch)\n        loaders["train"].sampler.set_epoch(epoch)\n        lr = scheduler.step(epoch)\n        if device.type == "cuda":\n            torch.cuda.reset_peak_memory_stats(device)\n        epoch_started = time.monotonic()\n        try:\n            stats, global_step = train_one_epoch(\n                model, loaders["train"], optimizer, device, scaler, config,\n                criterion, ema=ema, epoch=epoch, global_optimizer_step=global_step,\n            )\n            ema.module.set_epoch_temperature(epoch)\n            validation = evaluate_raw_and_tta(\n                ema.module, loaders["val"], device, config.use_amp\n            )\n        except Exception:\n            latest_resume = output / "resume_latest.pt"\n            failed_sha = sha256_file(latest_resume) if latest_resume.exists() else ""\n            _segment_manifest(\n                output, run_id=run_id, segment_number=config.segment_number,\n                start_epoch=segment_start_epoch, end_epoch=epoch - 1,\n                next_epoch=epoch, wallclock=time.monotonic() - segment_started,\n                resume_sha=failed_sha, best_epoch=best_epoch,\n                best_metrics=best_comparator, status="FAILED",\n            )\n            raise\n        duration = time.monotonic() - epoch_started\n        tta = validation["tta"]\n        improved = is_better_checkpoint(tta, best_comparator)\n        if improved:\n            best_comparator = dict(tta); best_metrics = {"raw": validation["raw"], "tta": tta}\n            best_epoch = epoch\n            _save_best_ema(checkpoint_path, ema, epoch, best_metrics, config, source_hash)\n        patience = update_early_stop_patience(epoch, improved, patience, config)\n        entry = {\n            "epoch": epoch, "lr": lr, "epoch_duration_sec": duration,\n            "val_raw": validation["raw"], "val_tta": tta,\n            "global_optimizer_step": global_step,\n            "early_stop_monitor_active": epoch >= config.early_stop_monitor_start_epoch,\n            "early_stop_patience": patience,\n            "gpu_peak_allocated_mib": torch.cuda.max_memory_allocated(device) / 2**20 if device.type == "cuda" else 0.0,\n            "gpu_peak_reserved_mib": torch.cuda.max_memory_reserved(device) / 2**20 if device.type == "cuda" else 0.0,\n            **stats,\n        }\n        history.append(entry); end_epoch = epoch; _write_history(history, output)\n        print(\n            f"Epoch {epoch:03d}/{config.max_epochs} [{duration:.1f}s] LR={lr:.8f} "\n            f"TrainAcc={stats[\'train_accuracy\']:.4f} RawAcc={validation[\'raw\'][\'accuracy\']:.4f} "\n            f"EMATTAAcc={tta[\'accuracy\']:.4f} EMATTAF1={tta[\'macro_f1\']:.4f} "\n            f"tau={stats[\'tau\']:.4f} Hlocal={stats[\'H_local_normalized\']:.4f} "\n            f"Hglobal={stats[\'H_global_normalized\']:.4f} alpha="\n            f"({stats[\'mean_alpha_8\']:.3f},{stats[\'mean_alpha_12\']:.3f},{stats[\'mean_alpha_16\']:.3f}) "\n            f"ConsFrac={stats[\'consistency_group_fraction\']:.3f} "\n            f"SupCon={stats[\'supcon_loss_raw\']:.4f} "\n            f"SupConValid={stats[\'valid_supcon_anchor_fraction\']:.3f}"\n        )\n\n        consistency_state = {\n            "algorithm": "stateless_seed_epoch_group_v1",\n            "last_completed_epoch": epoch,\n            "selected_groups": stats["consistency_groups"],\n        }\n        bundle = build_resume_bundle(\n            config=config, run_id=run_id, source_hash=source_hash,\n            completed_epoch=epoch, global_optimizer_step=global_step,\n            model=model, ema=ema, optimizer=optimizer, scheduler=scheduler,\n            scaler=scaler, best_comparator_state=best_comparator,\n            best_epoch=best_epoch, best_metrics=best_metrics,\n            early_stop_counter=patience, history=history,\n            loader_generator=generator, consistency_state=consistency_state,\n            sampler_state=loaders["train"].sampler.state_dict(),\n            augmentation_state=loaders["train"].dataset.state_dict(),\n        )\n        latest, resume_digest = atomic_save_resume(bundle, output)\n        save_periodic_snapshot(\n            latest, epoch, config.resume_snapshot_interval,\n            config.resume_snapshots_to_keep,\n        )\n        if should_early_stop(epoch, patience, config):\n            print(f"Early stopping triggered at epoch {epoch}")\n            break\n        elapsed = time.monotonic() - segment_started\n        estimate = max(item["epoch_duration_sec"] for item in history[-3:])\n        if (\n            epoch < config.max_epochs\n            and should_end_segment(elapsed, estimate, config)\n        ):\n            status = "NEEDS_RESUME"\n            _, resume_digest = atomic_save_resume(bundle, output, status=status)\n            break\n\n    elapsed = time.monotonic() - segment_started\n    if end_epoch >= segment_start_epoch and status == "TRAINING_COMPLETED":\n        bundle = build_resume_bundle(\n            config=config, run_id=run_id, source_hash=source_hash,\n            completed_epoch=end_epoch, global_optimizer_step=global_step,\n            model=model, ema=ema, optimizer=optimizer, scheduler=scheduler,\n            scaler=scaler, best_comparator_state=best_comparator,\n            best_epoch=best_epoch, best_metrics=best_metrics,\n            early_stop_counter=patience, history=history,\n            loader_generator=generator,\n            consistency_state={"algorithm": "stateless_seed_epoch_group_v1", "last_completed_epoch": end_epoch},\n            sampler_state=loaders["train"].sampler.state_dict(),\n            augmentation_state=loaders["train"].dataset.state_dict(),\n        )\n        _, resume_digest = atomic_save_resume(bundle, output, status=status)\n    segment = _segment_manifest(\n        output, run_id=run_id, segment_number=config.segment_number,\n        start_epoch=segment_start_epoch, end_epoch=end_epoch,\n        next_epoch=end_epoch + 1, wallclock=elapsed, resume_sha=resume_digest,\n        best_epoch=best_epoch, best_metrics=best_comparator, status=status,\n    )\n    result = {\n        "run_id": run_id, "source_hash": source_hash, "status": status,\n        "best_epoch": best_epoch, "best_metrics": best_metrics,\n        "best_checkpoint": str(checkpoint_path.resolve()) if checkpoint_path.exists() else None,\n        "best_checkpoint_sha256": sha256_file(checkpoint_path) if checkpoint_path.exists() else None,\n        "resume_checkpoint": str((output / "resume_latest.pt").resolve()),\n        "resume_sha256": resume_digest, "resumed_from": resumed_from,\n        "segment": segment, "PRIVATE_EVALUATED": False,\n    }\n    _atomic_json_document(output / "execution_manifest.json", result)\n    return result\n\n\ndef evaluate_private_once(\n    test_csv: str | Path, output_dir: str | Path, config: MPGConfig | None = None,\n) -> dict:\n    config = config or MPGConfig()\n    output = Path(output_dir)\n    manifest_path = output / "execution_manifest.json"\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n    if manifest["status"] != "TRAINING_COMPLETED":\n        raise RuntimeError("Private evaluation refused before training completion")\n    if manifest.get("PRIVATE_EVALUATED"):\n        raise RuntimeError("PrivateTest one-shot already recorded")\n    checkpoint_path = Path(manifest["best_checkpoint"])\n    if sha256_file(checkpoint_path) != manifest["best_checkpoint_sha256"]:\n        raise RuntimeError("Frozen EMA checkpoint hash mismatch")\n    loader = create_private_dataloader(test_csv, config.batch_size, config.num_workers)\n    device = torch.device(config.device if torch.cuda.is_available() else "cpu")\n    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)\n    state_tau = checkpoint["model_state_dict"].get("motif_composer.current_tau")\n    if state_tau is None or not math.isclose(\n        float(state_tau), float(checkpoint["scheduled_tau"]), rel_tol=0.0, abs_tol=1e-7\n    ):\n        raise RuntimeError("Best checkpoint scheduled temperature mismatch")\n    model = MPGFER(config).to(device); model.load_state_dict(checkpoint["model_state_dict"], strict=True)\n    metrics = evaluate_raw_and_tta(model, loader, device, config.use_amp)\n    _atomic_json_document(output / "private_metrics.json", metrics)\n    manifest["private_metrics"] = metrics\n    manifest["private_evaluated_only_after_freeze"] = True\n    manifest["PRIVATE_EVALUATED"] = True\n    _atomic_json_document(manifest_path, manifest)\n    return metrics\n', 'kaggle.py': '"""Fail-closed Kaggle data and explicit MPG-FER v2.2 resume resolution."""\n\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\nimport json\n\nfrom .checkpoint import find_latest_valid_snapshot, sha256_file\n\nfrom .data import validate_split_paths\n\n\nDATASET_SLUG = "doduyquynii/fer13-split"\nATTACHED_ROOTS = (\n    Path("/kaggle/input/datasets/doduyquynii/fer13-split/fer13-split"),\n    Path("/kaggle/input/fer13-split/fer13-split"),\n    Path("/kaggle/input/fer13-split"),\n)\n\n\ndef _complete_split_triplet(root: Path) -> tuple[Path, Path, Path] | None:\n    paths = (root / "train.csv", root / "val.csv", root / "test.csv")\n    return paths if all(path.is_file() for path in paths) else None\n\n\ndef resolve_kaggle_splits(\n    download_root: str | Path = "/kaggle/working/fer13-split-download",\n) -> tuple[Path, Path, Path]:\n    """Use an attached dataset or download it with Kaggle Secrets.\n\n    Credentials are read only from ``KAGGLE_USERNAME`` and ``KAGGLE_KEY``\n    secrets. The key is never printed or written to ``kaggle.json``.\n    """\n    for root in ATTACHED_ROOTS:\n        triplet = _complete_split_triplet(root)\n        if triplet is not None:\n            return validate_split_paths(*triplet)  # type: ignore[return-value]\n\n    try:\n        from kaggle_secrets import UserSecretsClient\n\n        secrets = UserSecretsClient()\n        username = secrets.get_secret("KAGGLE_USERNAME")\n        key = secrets.get_secret("KAGGLE_KEY")\n    except Exception as exc:\n        raise RuntimeError(\n            "FER13 split is not attached. Add Kaggle Secrets KAGGLE_USERNAME "\n            "and KAGGLE_KEY to download doduyquynii/fer13-split."\n        ) from exc\n    if not username or not key:\n        raise RuntimeError(\n            "FER13 split is not attached and Kaggle credentials are incomplete"\n        )\n\n    os.environ["KAGGLE_USERNAME"] = username\n    os.environ["KAGGLE_KEY"] = key\n    target = Path(download_root)\n    target.mkdir(parents=True, exist_ok=True)\n    try:\n        from kaggle.api.kaggle_api_extended import KaggleApi\n\n        api = KaggleApi()\n        api.authenticate()\n        api.dataset_download_files(DATASET_SLUG, path=target, unzip=True, quiet=False)\n    except Exception as exc:\n        raise RuntimeError(f"Failed to download Kaggle dataset {DATASET_SLUG}") from exc\n\n    candidates = [target, target / "fer13-split"]\n    candidates.extend(path.parent for path in target.rglob("train.csv"))\n    for root in candidates:\n        triplet = _complete_split_triplet(root)\n        if triplet is not None:\n            return validate_split_paths(*triplet)  # type: ignore[return-value]\n    raise RuntimeError(\n        f"Downloaded {DATASET_SLUG}, but train.csv/val.csv/test.csv were not found together"\n    )\n\n\ndef resolve_resume_artifact(\n    mode: str = "auto",\n    explicit_path: str | Path | None = None,\n    input_root: str | Path = "/kaggle/input",\n) -> tuple[Path | None, str | None]:\n    """Resolve only an explicit or clearly named v2.2 resume attachment.\n\n    Auto-discovery is deliberately restricted to datasets whose directory name\n    starts with ``mpg-fer-v2-2-resume``. It cannot discover v1/v2/v2.1\n    checkpoints or a generic ``best_val_acc.pt``.\n    """\n    normalized = mode.lower()\n    if normalized not in {"auto", "fresh", "required"}:\n        raise ValueError("RESUME_MODE must be auto, fresh, or required")\n    if normalized == "fresh":\n        return None, None\n    if explicit_path is not None:\n        candidates = [Path(explicit_path)]\n    else:\n        root = Path(input_root)\n        candidates = sorted(\n            path\n            for path in root.rglob("resume_latest.pt")\n            if any(\n                part.startswith("mpg-fer-v2-2-resume")\n                for part in path.relative_to(root).parts[:-1]\n            )\n        ) if root.exists() else []\n    if not candidates:\n        if normalized == "required":\n            raise FileNotFoundError("No explicit MPG-FER v2.2 resume artifact attached")\n        return None, None\n    if len(candidates) != 1:\n        raise RuntimeError(\n            f"Expected exactly one v2.2 resume artifact, found {len(candidates)}"\n        )\n    checkpoint = candidates[0]\n    if checkpoint.name != "resume_latest.pt" or not checkpoint.is_file():\n        raise RuntimeError(f"Invalid v2.2 resume checkpoint path: {checkpoint}")\n    metadata_path = checkpoint.with_name("resume_latest.json")\n    if not metadata_path.is_file():\n        raise RuntimeError("Resume metadata resume_latest.json is required")\n    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n    expected = metadata.get("checkpoint_sha256")\n    actual = sha256_file(checkpoint)\n    if not expected or actual != expected:\n        fallback = find_latest_valid_snapshot(\n            checkpoint.parent,\n            expected_identity=metadata,\n        )\n        detail = (\n            "none"\n            if fallback is None\n            else f"{fallback[0]} (sha256={fallback[1]})"\n        )\n        raise RuntimeError(\n            "Attached resume checkpoint SHA-256 mismatch; refusing automatic "\n            f"fallback. Most recent valid immutable snapshot: {detail}"\n        )\n    return checkpoint, actual\n', '__init__.py': '"""Root package for the Issue #95 MPG-FER v2.2 implementation."""\n\nfrom __future__ import annotations\n\nfrom .config import MPGConfig\nfrom .features import PixelFeatureExtractor\nfrom .graph import PixelGraphTopology\nfrom .motif import SpatialMotifComposer\nfrom .model import MPGFER\nfrom .ema import ModelEMA\n\n__all__ = [\n    "MPGConfig",\n    "PixelFeatureExtractor",\n    "PixelGraphTopology",\n    "SpatialMotifComposer",\n    "MPGFER",\n    "ModelEMA",\n]\n'}
EMBEDDED_ROOT = Path("/kaggle/working/mpg_fer_v2_2_embedded")
EMBEDDED_PACKAGE = EMBEDDED_ROOT / "mpg_fer_v2_2"
EMBEDDED_PACKAGE.mkdir(parents=True, exist_ok=True)
for relative_name, source_text in EMBEDDED_SOURCES.items():
    (EMBEDDED_PACKAGE / relative_name).write_text(source_text, encoding="utf-8")
sys.path.insert(0, str(EMBEDDED_ROOT))
print(f"Materialized reviewed MPG-FER v2.2 package at {EMBEDDED_PACKAGE}")


In [ ]:
# Environment, source/model/data gates, and explicit v2.2-only resume resolution.
from dataclasses import asdict
from datetime import datetime, timezone
import hashlib
import json
import math
import os
from pathlib import Path
import shutil
import time
import torch
import torch.nn as nn
from torch.optim import AdamW
import torchvision.transforms.functional as TF

from mpg_fer_v2_2.checkpoint import config_hash, load_resume_bundle, sha256_file
from mpg_fer_v2_2.config import MPGConfig
from mpg_fer_v2_2.data import FER2013Dataset, validate_dataset_gate
from mpg_fer_v2_2.ema import ModelEMA
from mpg_fer_v2_2.kaggle import resolve_kaggle_splits, resolve_resume_artifact
from mpg_fer_v2_2.model import MPGFER
from mpg_fer_v2_2.train import (
    compute_training_loss, evaluate_private_once, run_micro_overfit_preflight,
    run_training, source_tree_hash,
)
from mpg_fer_v2_2.utils import set_seed

def atomic_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
        handle.write("\n")
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
cfg = MPGConfig(segment_number=SEGMENT_NUMBER, output_dir=str(output_dir), resume_path=RESUME_PATH)
cuda_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_available else "NONE"
environment = {
    "torch_cuda_available": cuda_available,
    "gpu_name": gpu_name,
    "cuda_version": torch.version.cuda,
    "torch_version": torch.__version__,
}
print(json.dumps(environment, indent=2))
if not cuda_available or "t4" not in gpu_name.lower():
    atomic_json(output_dir / "preflight_report.json", {
        "status": "WRONG_GPU", "environment": environment,
        "kernel_ref": KERNEL_REF, "segment_number": SEGMENT_NUMBER,
    })
    raise RuntimeError(f"WRONG_GPU: expected NVIDIA Tesla T4, got {gpu_name}")
device = torch.device("cuda")

reviewed_source_sha = source_tree_hash()
if reviewed_source_sha != EXPECTED_SOURCE_SHA:
    raise RuntimeError(
        f"SOURCE_HASH_MISMATCH: expected {EXPECTED_SOURCE_SHA}, got {reviewed_source_sha}"
    )
model_for_count = MPGFER(cfg)
parameter_count = sum(p.numel() for p in model_for_count.parameters() if p.requires_grad)
del model_for_count
if parameter_count != EXPECTED_PARAMETERS:
    raise RuntimeError(
        f"PARAMETER_COUNT_MISMATCH: expected {EXPECTED_PARAMETERS}, got {parameter_count}"
    )

train_csv, val_csv, test_csv = resolve_kaggle_splits()
dataset_gate = validate_dataset_gate(train_csv, val_csv, test_csv)
resume_checkpoint, resume_sha = resolve_resume_artifact(RESUME_MODE, RESUME_PATH)

# The only allowed scientific fallback is selected before the fresh official run.
# A resumed bundle declares which of the two preregistered batch/accumulation pairs it used.
if resume_checkpoint is not None:
    inspected = torch.load(resume_checkpoint, map_location="cpu", weights_only=False)
    resumed_scientific = inspected.get("scientific_config", {})
    resumed_pair = (
        int(resumed_scientific.get("batch_size", cfg.batch_size)),
        int(resumed_scientific.get("gradient_accumulation_steps", cfg.gradient_accumulation_steps)),
    )
    if resumed_pair == (8, 4):
        cfg.batch_size, cfg.gradient_accumulation_steps = resumed_pair
    elif resumed_pair != (16, 2):
        raise RuntimeError(f"Forbidden resumed batch/accumulation pair: {resumed_pair}")
    del inspected

print(json.dumps({
    "account": ACCOUNT, "kernel_ref": KERNEL_REF, "git_commit": GIT_COMMIT_SHA,
    "segment_number": SEGMENT_NUMBER, "source_sha256": reviewed_source_sha,
    "parameter_count": parameter_count, "dataset": dataset_gate,
    "resume_checkpoint": None if resume_checkpoint is None else str(resume_checkpoint),
}, indent=2))


In [ ]:
# Fresh T4 preflight or strict resume-state verification.
def run_execution_bounded_preflight(train_path, config, device):
    set_seed(config.seed)
    dataset = FER2013Dataset(train_path, split="train", augment=False)
    images = torch.stack([dataset[index][0] for index in range(config.batch_size)]).to(device)
    targets = torch.tensor(
        [dataset[index][1] for index in range(config.batch_size)],
        dtype=torch.long, device=device,
    )
    del dataset
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)
    model = MPGFER(config).to(device).train()
    model.set_epoch_temperature(1)
    ema = ModelEMA(model, config.ema_decay)
    optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    scaler = torch.amp.GradScaler("cuda")
    criterion = nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)
    finite = {}
    hooks = []

    def finite_hook(name):
        def record(_module, _inputs, output):
            finite[name] = bool(torch.isfinite(output).all().item())
        return record

    hooks.append(model.pixel_extractor.register_forward_hook(finite_hook("features")))
    for index, layer in enumerate(model.pixel_gnn):
        hooks.append(layer.attn_dropout.register_forward_hook(finite_hook(f"pixel_attention_{index}")))
    for index, layer in enumerate(model.motif_gnn):
        hooks.append(layer.attn_dropout.register_forward_hook(finite_hook(f"motif_attention_{index}")))

    optimizer.zero_grad(set_to_none=True)
    torch.cuda.synchronize(device)
    step_started = time.perf_counter()
    with torch.amp.autocast("cuda", enabled=True):
        logits, outputs = model(images)
        flipped_logits, flipped_outputs = model(TF.hflip(images))
        loss, components = compute_training_loss(
            logits, outputs, targets, criterion, config, flipped_logits
        )
    tensors = {
        "final_logits": logits, "flipped_logits": flipped_logits,
        "motif_assignments": outputs["motif_assignments"],
        "motif_geometry": outputs["motif_geometry"],
        "temperature": outputs["tau"], "mi_loss": outputs["loss_mi"],
        "consistency_loss": components["weighted_consistency"],
        "supcon_loss": components["supcon_loss_raw"],
        "supcon_embeddings": outputs["supcon_embeddings"],
        "total_loss": loss,
    }
    finite.update({name: bool(torch.isfinite(value).all().item()) for name, value in tensors.items()})
    if not all(finite.values()):
        raise FloatingPointError(f"Non-finite T4 preflight tensor: {finite}")
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    audited = {
        "pixel_projection": model.pixel_proj[0].weight,
        "pixel_gnn": model.pixel_gnn[0].q_proj.weight,
        "assignment_query": model.motif_composer.assignment_query.weight,
        "prototypes": model.motif_composer.prototypes,
        "scale_gate": model.motif_composer.scale_gate.weight,
        "scale_saliency_8": model.motif_composer.scale_saliency["8"].weight,
        "motif_geometry_attention": model.motif_gnn[0].geom_proj.weight,
        "pixel_readout": model.pixel_readout_proj[0].weight,
        "motif_readout": model.motif_readout_proj[0].weight,
        "supcon_projection": model.supcon_head[0].weight,
        "classifier": model.classifier[-1].weight,
    }
    gradients = {
        name: bool(
            parameter.grad is not None
            and torch.isfinite(parameter.grad).all().item()
            and torch.any(parameter.grad != 0).item()
        )
        for name, parameter in audited.items()
    }
    if not all(gradients.values()):
        raise FloatingPointError(f"Invalid T4 preflight gradient: {gradients}")
    torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
    old_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    optimizer_step_succeeded = scaler.get_scale() >= old_scale
    if not optimizer_step_succeeded:
        raise FloatingPointError("GradScaler skipped the bounded optimizer step")
    ema.update(model)
    torch.cuda.synchronize(device)
    step_time_seconds = time.perf_counter() - step_started
    result = {
        "passed": True,
        "physical_batch": config.batch_size,
        "gradient_accumulation": config.gradient_accumulation_steps,
        "effective_batch": config.batch_size * config.gradient_accumulation_steps,
        "amp_enabled": True,
        "worst_case_consistency_forward": True,
        "supcon_enabled": True,
        "loss": float(loss.detach()),
        "loss_components": {name: float(value.detach()) for name, value in components.items()},
        "finite_audits": finite,
        "gradient_audits": gradients,
        "ema_updates": ema.num_updates,
        "step_time_seconds": step_time_seconds,
        "motif_diagnostics": {
            name: float(outputs[name].detach())
            for name in (
                "tau", "H_local_raw", "H_local_normalized", "H_global_raw",
                "H_global_normalized", "L_MI", "mean_entropy",
                "effective_motif_count", "min_utilization", "max_utilization",
                "std_utilization", "mean_top1_probability", "mean_top2_probability",
                "mean_top1_top2_margin", "mean_offdiag_prototype_cosine",
                "mean_alpha_8", "mean_alpha_12", "mean_alpha_16",
            )
        },
        "peak_allocated_mib": torch.cuda.max_memory_allocated(device) / 2**20,
        "peak_reserved_mib": torch.cuda.max_memory_reserved(device) / 2**20,
    }
    for hook in hooks:
        hook.remove()
    del model, ema, optimizer, scaler, images, targets, logits, flipped_logits, outputs, flipped_outputs, loss
    torch.cuda.empty_cache()
    return result

resume_verification = None
if resume_checkpoint is None:
    try:
        bounded = run_execution_bounded_preflight(train_csv, cfg, device)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        cfg.batch_size = 8
        cfg.gradient_accumulation_steps = 4
        bounded = run_execution_bounded_preflight(train_csv, cfg, device)
        bounded["oom_fallback_applied"] = True
    micro = run_micro_overfit_preflight(
        train_csv, cfg, device=device, raise_on_failure=False
    )
    if not micro["passed"]:
        preflight = {
            "status": "PREFLIGHT_FAILED", "passed": False,
            "environment": environment, "dataset": dataset_gate,
            "bounded_gpu": bounded, "micro_overfit": micro,
        }
        atomic_json(output_dir / "preflight_report.json", preflight)
        raise RuntimeError(f"REAL_MICRO_OVERFIT_FAILED: {micro}")
    preflight = {
        **micro, "status": "PASS", "environment": environment,
        "dataset": dataset_gate, "parameter_count": parameter_count,
        "source_sha256": reviewed_source_sha, "bounded_gpu": bounded,
        "fresh_reset_ready": True,
    }
else:
    metadata = json.loads(resume_checkpoint.with_name("resume_latest.json").read_text(encoding="utf-8"))
    bundle = load_resume_bundle(
        resume_checkpoint, expected_sha256=resume_sha, config=cfg,
        run_id=metadata["run_id"], source_hash=reviewed_source_sha,
    )
    previous_segment_path = resume_checkpoint.with_name("segment_manifest.json")
    if not previous_segment_path.is_file():
        raise RuntimeError("Attached resume artifact lacks segment_manifest.json")
    previous_segment = json.loads(previous_segment_path.read_text(encoding="utf-8"))
    checks = {
        "run_id": bundle["run_id"] == metadata["run_id"] == previous_segment["run_id"],
        "source_hash": bundle["source_hash"] == metadata["source_hash"] == reviewed_source_sha,
        "config_hash": bundle["config_hash"] == metadata["config_hash"] == config_hash(cfg),
        "epoch_boundary": bundle["next_epoch"] == previous_segment["next_epoch"],
        "history_length": len(bundle["history"]) == bundle["completed_epoch"],
        "previous_status": previous_segment["status"] == "NEEDS_RESUME",
    }
    if not all(checks.values()):
        raise RuntimeError(f"RESUME_STATE_MISMATCH: {checks}")
    best_source = resume_checkpoint.with_name("best_val_acc.pt")
    best_metadata_path = resume_checkpoint.with_name("best_val_acc.json")
    if bundle["best_epoch"] is not None:
        if not best_source.is_file() or not best_metadata_path.is_file():
            raise RuntimeError("Resume artifact lacks globally best EMA checkpoint/metadata")
        best_metadata = json.loads(best_metadata_path.read_text(encoding="utf-8"))
        best_checks = {
            "sha256": sha256_file(best_source) == best_metadata["checkpoint_sha256"],
            "run_id": best_metadata["run_id"] == bundle["run_id"],
            "source_hash": best_metadata["source_hash"] == reviewed_source_sha,
            "config_hash": best_metadata["config_hash"] == bundle["config_hash"],
            "best_epoch": best_metadata["best_epoch"] == bundle["best_epoch"],
        }
        if not all(best_checks.values()):
            raise RuntimeError(f"BEST_CHECKPOINT_RESUME_MISMATCH: {best_checks}")
        shutil.copy2(best_source, output_dir / "best_val_acc.pt")
    restored_tau = bundle["temperature_state"]["current_tau"]
    scaler_state = bundle["grad_scaler_state_dict"]
    resume_verification = {
        "checks": checks,
        "run_id": bundle["run_id"],
        "segment_number": SEGMENT_NUMBER,
        "source_hash": bundle["source_hash"],
        "scientific_config_hash": bundle["config_hash"],
        "completed_epoch": bundle["completed_epoch"],
        "next_epoch": bundle["next_epoch"],
        "restored_lr": bundle["optimizer_state_dict"]["param_groups"][0]["lr"],
        "restored_optimizer_step": bundle["global_optimizer_step"],
        "restored_grad_scaler_scale": scaler_state.get("scale"),
        "restored_ema_updates": bundle["ema_state_dict"]["num_updates"],
        "restored_tau": restored_tau,
        "best_epoch": bundle["best_epoch"],
        "best_public_tta_accuracy": None if bundle["best_comparator_state"] is None else bundle["best_comparator_state"]["accuracy"],
        "patience_counter": bundle["early_stop_counter"],
        "early_stop_monitor_state": bundle["early_stop_monitor_state"],
        "history_length": len(bundle["history"]),
    }
    print("RESUME_VERIFICATION")
    print(json.dumps(resume_verification, indent=2))
    preflight = {
        "status": "RESUME_VERIFIED", "resume_compatibility_passed": True,
        "resume_checkpoint_sha256": resume_sha, "resume_verification": resume_verification,
        "environment": environment, "dataset": dataset_gate,
        "parameter_count": parameter_count, "source_sha256": reviewed_source_sha,
    }
    del bundle

atomic_json(output_dir / "preflight_report.json", preflight)
print(json.dumps(preflight, indent=2))


In [ ]:
# Fresh official run or exact continuation from next_epoch.
manifest = run_training(
    train_csv=train_csv, val_csv=val_csv, output_dir=output_dir,
    preflight_result=preflight, config=cfg,
    resume_path=resume_checkpoint, resume_sha256=resume_sha,
)

# Complete the segment provenance contract and preserve the global best EMA checkpoint.
resume_metadata = json.loads((output_dir / "resume_latest.json").read_text(encoding="utf-8"))
segment_path = output_dir / "segment_manifest.json"
segment_manifest = json.loads(segment_path.read_text(encoding="utf-8"))
segment_manifest.update({
    "source_hash": resume_metadata["source_hash"],
    "config_hash": resume_metadata["config_hash"],
    "gpu": gpu_name,
    "kernel_ref": KERNEL_REF,
    "git_commit_sha": GIT_COMMIT_SHA,
    "parameter_count": parameter_count,
    "resume_verification": resume_verification,
})
atomic_json(segment_path, segment_manifest)

best_path = output_dir / "best_val_acc.pt"
if not best_path.is_file():
    raise RuntimeError("Global best EMA checkpoint is missing at segment end")
best_metadata = {
    "checkpoint_sha256": sha256_file(best_path),
    "weights_type": "EMA",
    "run_id": manifest["run_id"],
    "source_hash": resume_metadata["source_hash"],
    "config_hash": resume_metadata["config_hash"],
    "best_epoch": manifest["best_epoch"],
    "best_public_tta_accuracy": manifest["best_metrics"]["tta"]["accuracy"],
    "scheduled_tau": torch.load(best_path, map_location="cpu", weights_only=False)["scheduled_tau"],
}
atomic_json(output_dir / "best_val_acc.json", best_metadata)
print(json.dumps({"manifest": manifest, "segment_manifest": segment_manifest,
                  "best_checkpoint": best_metadata}, indent=2))


In [ ]:
# Final metrics/artifacts only after TRAINING_COMPLETED; otherwise clean resume archive.
if manifest["status"] == "TRAINING_COMPLETED":
    private_metrics = evaluate_private_once(test_csv, output_dir, cfg)
    execution_manifest = json.loads((output_dir / "execution_manifest.json").read_text(encoding="utf-8"))
    public_metrics = execution_manifest["best_metrics"]
    atomic_json(output_dir / "public_metrics.json", public_metrics)

    history = json.loads((output_dir / "history.json").read_text(encoding="utf-8"))
    best_entry = next(entry for entry in history if entry["epoch"] == execution_manifest["best_epoch"])
    motif_names = [
        "tau", "H_local_raw", "H_local_normalized", "H_global_raw",
        "H_global_normalized", "L_MI", "mean_entropy", "effective_motif_count",
        "min_utilization", "max_utilization", "std_utilization",
        "mean_top1_probability", "mean_top2_probability",
        "mean_top1_top2_margin", "mean_offdiag_prototype_cosine",
        "mean_alpha_8", "mean_alpha_12", "mean_alpha_16",
        "std_alpha_8", "std_alpha_12", "std_alpha_16",
    ]
    motif_diagnostics = {
        "best_epoch": execution_manifest["best_epoch"],
        "best_epoch_diagnostics": {name: best_entry[name] for name in motif_names},
        "final_epoch": history[-1]["epoch"],
        "final_epoch_diagnostics": {name: history[-1][name] for name in motif_names},
    }
    atomic_json(output_dir / "motif_diagnostics.json", motif_diagnostics)
    selection = {
        "run_id": execution_manifest["run_id"],
        "weights_type": "EMA",
        "best_epoch": execution_manifest["best_epoch"],
        "checkpoint_sha256": sha256_file(output_dir / "best_val_acc.pt"),
        "selection_metric": "EMA Public flip-TTA accuracy",
        "selection_tiebreak": ["higher EMA TTA macro-F1", "lower EMA TTA loss"],
        "private_evaluated_only_after_freeze": True,
    }
    atomic_json(output_dir / "final_selection_manifest.json", selection)

    import matplotlib.pyplot as plt
    def save_confusion(metrics, path, title):
        matrix = metrics["confusion_matrix"]
        figure, axis = plt.subplots(figsize=(7, 6))
        image = axis.imshow(matrix, cmap="Blues")
        axis.set(title=title, xlabel="Predicted", ylabel="True")
        axis.set_xticks(range(7)); axis.set_yticks(range(7))
        for row in range(7):
            for column in range(7):
                axis.text(column, row, str(matrix[row][column]), ha="center", va="center")
        figure.colorbar(image, ax=axis); figure.tight_layout(); figure.savefig(path, dpi=160); plt.close(figure)

    save_confusion(public_metrics["tta"], output_dir / "public_confusion_matrix.png", "PublicTest EMA TTA")
    save_confusion(private_metrics["tta"], output_dir / "private_confusion_matrix.png", "PrivateTest EMA TTA")
    epochs = [entry["epoch"] for entry in history]
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, [entry["train_loss"] for entry in history], label="train")
    axes[0].plot(epochs, [entry["val_tta"]["loss"] for entry in history], label="EMA Public TTA")
    axes[1].plot(epochs, [entry["train_accuracy"] for entry in history], label="train")
    axes[1].plot(epochs, [entry["val_tta"]["accuracy"] for entry in history], label="EMA Public TTA")
    for axis in axes:
        axis.legend(); axis.grid(alpha=0.2); axis.set_xlabel("Epoch")
    axes[0].set_title("Loss"); axes[1].set_title("Accuracy")
    figure.tight_layout(); figure.savefig(output_dir / "training_curves.png", dpi=160); plt.close(figure)
    print(json.dumps({"public": public_metrics, "private": private_metrics,
                      "motif": motif_diagnostics, "selection": selection}, indent=2))
else:
    print("Segment ended NEEDS_RESUME normally; PrivateTest remains unopened.")

archive = shutil.make_archive("/kaggle/working/mpg_fer_v2_2_artifacts", "zip", root_dir=output_dir)
print(f"Artifacts ZIP: {archive}")
